## 代码完整展示+设计文档设计+从头书写


---

# LedgerScope 代码库上下文 Prompt（完整版）

## 一、项目概述

LedgerScope 是一个**通用财务建模框架**，核心抽象包括 Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization。框架与业务解耦，可复用于外贸、电商、SaaS、制造业等场景。

### 当前版本约束
- 仅支持单一商品模型
- 订单来源假设为广告渠道（无复购、无分销）
- 转化率恒定
- 折旧与资本支出为占位（返回 0）
- 无订单交付时间滞后
- 固定汇率

### 未来规划
- 升级包模块、多渠道归因、批量折扣、多商品支持、复购与 LTV、汇率波动模拟等

---

## 二、core/ 基类

### 2.1 Variable 基类

**位置**: `src/core/base_variable.py`

**构造规则**（5 种情况）：

| 输入 | 处理 |
|:---|:---|
| min, exp, max 全提供 | 直接使用 |
| 仅 exp | min = max = exp |
| 仅 min + max | exp = (min + max) / 2 |
| 仅 max | min = 0, exp = max / 2 |
| 全空 | 全部 None（占位变量） |
| 仅 min | 语义不清晰，不允许 |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `get_value(ValueType)` | 根据策略返回对应值（EXPECTED/MIN/MAX/RANDOM） |
| `get_random_value()` | 返回 [min, max] 范围内的随机值（均匀分布） |
| `get_range_values(num)` | 返回等间距线性空间数组（numpy），用于扫描分析 |
| `set_value(value)` | 固定为常量（不推荐，会丢失范围信息） |

**重要属性**：
- `_name`: 变量名，由子类在 `__init__` 中设置，无 setter
- 子类变量名必须与 `config/variable_names.py` 中的常量对应

**设计决策**：
- 不提供默认业务边界：避免隐性假设，用户必须显式传入
- 实例不可变：保持可复现性，新场景应创建新实例

### 2.2 Model 基类

**位置**: `src/core/base_model.py`

**核心属性**：

| 属性 | 类型 | 说明 |
|:---|:---|:---|
| `_required_variables` | `list[str]` | 必需变量名，缺失时抛出 KeyError |
| `_optional_variables` | `dict[str, float]` | 可选变量名及其默认值 |
| `_model_function` | `callable` | 核心计算函数 |
| `_output_names` | `list[str]` | 输出变量名列表 |
| `_input_variables` | `dict` | 共享状态字典（就地更新） |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `check_variables()` | 验证必需变量存在（缺失抛 KeyError），可选变量缺失仅 log.info |
| `evaluate()` | 执行验证 → 调用 `_model_function` → update 结果 → 返回共享状态 |
| `update_input_variable(key, value)` | 支持 Variable 对象或 key-value 两种签名 |

**_model_function 签名规范**：

```python
def calculate_example(optional_variables: dict, **kwargs) -> dict:
    # optional_variables: 可选变量默认值字典
    # **kwargs: 运行时变量字典
    # 返回: {output_name: value}
```

**重要约定**：
- 可选变量通过 `kwargs.get(key, optional_variables[key])` 获取默认值
- 返回值必须是 `dict`，非 dict 时 `evaluate()` 跳过 update
- 建议添加除零保护
- 函数内不修改 `**kwargs`

**设计决策**：
- Model 不依赖 Variable 类：可独立测试，输入来源灵活
- 就地更新策略：内存优化 + 链式执行，调用方需自行 copy 保留原始状态

### 2.3 Auditor 基类

**位置**: `src/core/base_auditor.py`

**继承关系**：`Auditor` 继承 `Model`

**差异**：
- `output_names` 返回空列表（不产生新变量）
- `evaluate()` 调用 `_model_function` 后返回原字典（不更新）
- 验证失败时 `_model_function` 内部抛出 `ValueError`

**验证函数规范**：

```python
def check_xxx(optional_variables: dict, **kwargs) -> None:
    # 验证逻辑
    if not condition:
        raise ValueError("清晰的错误信息")
    # 验证通过，正常返回（无返回值）
```

**设计决策**：
- Auditor 是 Model 特化：可无缝嵌入 Pipeline，无需特殊处理
- 验证失败中断 Pipeline：快速失败原则

---

## 三、variables/ 变量定义

### 3.1 文件结构

| 文件 | 内容 |
|:---|:---|
| `advertising.py` | AdvertisingBudget, GoogleSearchConversionRate, GoogleSearchCostPerClick, GoogleSearchAllocationPercentage |
| `costs.py` | Cost, AdvertisingCost, SetupCost |
| `deals.py` | Orders, CloseRate, UnitExw, UnitRetail, ChannelMarkupRate, ShippingRate, DeductionRate, UnitFob, UnitsPerOrder |
| `expenses.py` | Expense, MonthlyExpense, RentExpense, TravelExpense, RenderExpense |
| `finance.py` | InterestRate, TaxRate, USDToRMB, TariffRate, PriceToEarningsRatio |

### 3.2 变量定义模板

```python
from src.config import variable_names
from src.core import Variable

class MyVariable(Variable):
    def __init__(self, min=None, exp=None, max=None):
        super().__init__(min, exp, max)
        self._name = variable_names.MY_VARIABLE
```

### 3.3 重要变量说明

| 变量 | 说明 |
|:---|:---|
| `SetupCost` | 一次性启动成本，作为**投资项**（不计入 TotalCost），用于 ROI 分母 |
| `DeductionRate` | 聚合扣率 = ShippingRate + TariffRate + ChannelMarkupRate |
| `UnitExw` | 出厂价（人民币） |
| `UnitFob` | FOB 价（美元），从零售价反推 |
| `UnitRetail` | 零售价（美元） |
| `USDToRMB` | 汇率默认 6.8 |

---

## 四、models/ 模型实现

### 4.1 模型分类与清单

**广告漏斗（advertising/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `AdvertisingEfficiencyGoogleSearchModel` | `Leads = (Budget × Allocation) / (CPC × USDToRMB) × CVR` | LEADS |
| `CostPerLeadGoogleSearchModel` | `CPL = CPC / (CVR × Allocation)` | CPL |

**成本（cost/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `CostOfGoodsSoldModel` | `COGS = UnitExw × Orders × UnitsPerOrder` | COGS |
| `ShippingCostModel` | `ShippingCost = UnitRetail × USDToRMB × ShippingRate × Orders × UnitsPerOrder` | SHIPPING_COST |
| `TotalCostModel` | `TotalCost = COGS + AdvertisingCost + ShippingCost`（不含 SetupCost）| COST |

**交易（deal/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `DeductionRateModel` | `DeductionRate = ShippingRate + TariffRate + ChannelMarkupRate` | DEDUCTION_RATE |
| `OrderModel` | `Orders = Leads × CloseRate` | ORDERS |
| `UnitFobModel` | `UnitFob = UnitRetail × (1 - DeductionRate)` | UNIT_FOB |
| `UnitContributionMarginModel` | `UnitContributionMargin = Profit / (Orders × UnitsPerOrder)` | UNIT_CONTRIBUTION_MARGIN |

**费用（expense/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `MonthlyExpenseModel` | `MonthlyExpense = Rent + Render + Travel` | MONTHLY_EXPENSE |
| `TotalExpenseModel` | `Expense = MonthlyExpense × Months` | EXPENSE |

**收入与利润（income/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `RevenueModel` | `Revenue = UnitFob × Orders × UnitsPerOrder × USDToRMB` | REVENUE |
| `ProfitModel` | `Profit = Revenue - Cost` | PROFIT |
| `NetIncomeModel` | `NetIncome = (Revenue - Cost - Expense - Depreciation) × (1 - TaxRate)` | NET_INCOME |
| `FreeCashFlowModel` | `FCF = NetIncome + Depreciation - CapEx` | FREE_CASH_FLOW |

**指标（metrics/）**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `CacModel` | `CAC = AdvertisingCost / Orders` | CAC |
| `RoasModel` | `ROAS = Revenue / AdvertisingCost` | ROAS |
| `RoiModel` | `ROI = NetIncome / SetupCost` | ROI |
| `MarketPriceModel` | `MarketPrice = (NetIncome × 12 × PE) / Months` | MARKET_PRICE |
| `PriceArchitectureModel` | 分解零售价为 5 个组成部分 | COGS_PER_UNIT, PROFIT_PER_UNIT, SHIPPING_COST_PER_UNIT, TARIFF_PER_UNIT, RETAIL_MARGIN_PER_UNIT |

**占位（finance/）**：

| 模型 | 输出 | 说明 |
|:---|:---|:---|
| `CapitalExpenditureModel` | 0 | 资本支出占位 |
| `DepreciationModel` | 0 | 折旧占位 |

### 4.2 模型实现模板

```python
from src.config import variable_names
from src.core import Model

def calculate_xxx(optional_variables: dict, **kwargs) -> dict:
    # 1. 读取必需变量（会抛出 KeyError）
    required_var = kwargs[variable_names.REQUIRED_VAR]
    
    # 2. 读取可选变量（带默认值）
    default_val = optional_variables[variable_names.OPTIONAL_VAR]
    optional_var = kwargs.get(variable_names.OPTIONAL_VAR, default_val)
    
    # 3. 除零保护
    if denominator == 0:
        return {variable_names.OUTPUT: 0.0}
    
    # 4. 计算
    result = ...
    
    # 5. 返回
    return {variable_names.OUTPUT: result}

class XxxModel(Model):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = calculate_xxx
        self._output_names = [variable_names.OUTPUT]
        self._required_variables = [variable_names.REQUIRED_VAR]
        self._optional_variables = {variable_names.OPTIONAL_VAR: 0.0}
```

### 4.3 模型依赖关系

数据流方向：
```
LEADS → ORDERS → COGS/REVENUE → COST → PROFIT → NET_INCOME → MARKET_PRICE/FCF/ROI
```

---

## 五、auditors/ 审计器

### 5.1 PriceArchitectureAuditor

**位置**: `auditors/price_architecture_auditor.py`

**验证规则**：
1. `COGS_per_unit + Profit_per_unit == UnitFob`
2. `UnitFob + ShippingCost_per_unit + Tariff_per_unit + RetailMargin_per_unit == UnitRetail`

**容差设置**：使用 `settings.AUDIT_REL_TOL` (1e-3) 和 `AUDIT_ABS_TOL` (1e-2)

### 5.2 Auditor 模板

```python
from src.core import Auditor

def check_xxx(optional_variables: dict, **kwargs):
    var = kwargs[variable_names.VAR]
    if not condition:
        raise ValueError(f"验证失败: {var}")

class XxxAuditor(Auditor):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = check_xxx
        self._required_variables = [variable_names.VAR]
```

---

## 六、pipelines/ 管道

### 6.1 model_registry.py

`PIPELINE_REGISTRY` 映射模型名称字符串到模型类，包含 22 个模型 + 审计器。

### 6.2 model_composer.py

`PipelineComposer` 方法：

| 方法 | 说明 |
|:---|:---|
| `build_pipeline_by_keys(keys)` | 根据模型名称列表实例化 |
| `build_named_scenario(name, *extra)` | 构建预定义场景，可追加模型 |
| `build_merged_scenarios(names)` | 合并多个场景，去重 |

### 6.3 预定义场景

位于 `config/pipelines.py`：
- `costs`: 成本分析
- `marketing_roi_analysis`: 营销 ROI
- `earnings_and_capex_run`: 资本与收入
- `complete_macro_metrics`: 完整宏观指标

---

## 七、engine/ 执行引擎

### 7.1 runner.py 函数

| 函数 | 说明 |
|:---|:---|
| `evaluate_chained_models(state, pipeline)` | 顺序执行模型，使用 deepcopy 隔离状态 |
| `evaluate_expected_scenario(variables, pipeline)` | 所有变量取期望值 |
| `evaluate_stochastic_iteration(variables, shuffled, pipeline)` | 单次随机采样 |
| `evaluate_variable_scenario_sweep(variables, var, values, pipeline)` | 单变量扫描 |

### 7.2 使用注意

- 每次执行都会 deepcopy 状态，确保模拟独立性
- 拓扑验证应在调用前由 `validation.py` 完成

---

## 八、analysis/ 分析框架

### 8.1 6 种分析模式

| 函数 | 用途 | 关键参数 |
|:---|:---|:---|
| `break_even_analysis` | 盈亏平衡 | goal（目标值） |
| `comparative_statics` | 三点敏感性 | 无 |
| `stochastic_contribution_analysis` | 贡献度饼图 | breakdown_metrics, shuffled_inputs |
| `run_monte_carlo` | 蒙特卡洛模拟 | iterations, tracked_outputs |
| `stochastic_bivariate_simulation` | 回归分析 | x, y, shuffled_variables |
| `run_two_way_sensitivity_analysis` | 双变量热力图 | x_steps, y_steps, reverse_y |

### 8.2 分析函数模板

```python
from src.engine import evaluate_expected_scenario
from src.utils import check_model_pipeline_topology_order

def new_analysis(variables, model_pipeline, output_name, **params):
    check_model_pipeline_topology_order(model_pipeline)
    baseline = evaluate_expected_scenario(variables, model_pipeline)
    # ... 分析逻辑 ...
    return {"baseline": baseline[output_name], ...}
```

---

## 九、visualization/ 可视化

### 9.1 视图与分析的对应

| 视图 | 对应分析 | 输出 |
|:---|:---|:---|
| `render_break_even_dashboard` | break_even_analysis | Styler 表格 |
| `render_comparative_statics_dashboard` | comparative_statics | Styler 表格 |
| `generate_contribution_pie_chart` | contribution_analysis | 饼图 |
| `generate_histogram_from_array` | monte_carlo | 直方图 |
| `generate_linear_regression_from_lists` | regression | 散点图 + 回归线 |
| `generate_heatmap_from_df` | two_way_sensitivity | 热力图 |

### 9.2 common_view 工具

| 函数 | 说明 |
|:---|:---|
| `get_formatter(var_name)` | 从 `VARIABLE_FORMATTING_MAP` 获取格式化函数 |
| `apply_custom_variable_formatting(row, var_col, target_cols)` | 对 DataFrame 行应用格式化 |

---

## 十、config/ 配置

### 10.1 settings.py

| 参数 | 默认值 | 说明 |
|:---|:---|:---|
| `NUMS_IN_RANGE` | 50 | 扫描步数 |
| `DECIMAL_ROUNDING` | 4 | 浮点精度 |
| `SAMPLE_SIZE` | 100 | 蒙特卡洛默认迭代（生产用 5000） |
| `AUDIT_REL_TOL` | 1e-3 | 审计相对容差 |
| `AUDIT_ABS_TOL` | 1e-2 | 审计绝对容差 |
| `DEFAULT_CURRENCY_RATE` | 6.8 | USD→RMB |
| `DEFAULT_TAX_RATE` | 0.2 | 税率 |
| `DEFAULT_TARIFF_RATE` | 0.25 | 关税率 |
| `DEFAULT_PE_RATIO` | 8 | 市盈率 |

### 10.2 variable_names.py

50+ 常量，分类：Expenses、Investment、Deals、Finance、Ads、Costs、Revenue、Metrics、Price Architecture、Break even、Comparative statics

### 10.3 formatting.py

`VARIABLE_FORMATTING_MAP` 定义每个变量的格式化规则：

```python
VARIABLE_FORMATTING_MAP = {
    "Revenue": lambda v: fmt(v, s='¥'),
    "CPC_GoogleSearch": lambda v: fmt(v, d=1, s='$'),
    "TaxRate": lambda v: fmt(v, d=2, p=True),
}
```

### 10.4 messages.py

日志和错误信息模板，包括：
- 拓扑顺序错误、变量构造错误、非单调关系等
- 盈亏平衡状态码（CROSSOVER_FOUND、UNREACHABLE、ALWAYS_FEASIBLE）

---

## 十一、utils/ 工具

### 11.1 validation.py

| 函数 | 说明 |
|:---|:---|
| `get_missing_elements(provided, required)` | 返回缺失变量列表 |
| `check_variables_for_function(provided, required)` | 缺失则抛出 KeyError |
| `check_model_pipeline_topology_order(models)` | 验证拓扑顺序（已消费变量不能重复产出） |

### 11.2 formatting.py

```python
def fmt(val, d=0, s='', p=False) -> str:
    # d: 小数位数
    # s: 货币符号前缀
    # p: 百分比模式（乘以 100）
```

### 11.3 logger.py

配置开关：
- `WRITE_TO_FILE = False`
- `PRINT_TO_CONSOLE = True`
- `LOG_LEVEL = logging.ERROR`（调试时改为 INFO/DEBUG）

---

## 十二、关键设计决策汇总

| 决策 | 原因 |
|:---|:---|
| Variable 无默认边界 | 避免隐性假设 |
| Variable 不可变 | 保持可复现性 |
| Model 不依赖 Variable | 可独立测试 |
| Auditor 是 Model 特化 | 统一接口，无缝嵌入 |
| 拓扑顺序验证（DAG） | 防止变量覆盖冲突 |
| SetupCost 不计入 TotalCost | 作为投资项，用于 ROI |
| 订单单一来源 | 简化 MVP |

---

## 十三、已知 Bug 与修复

| Bug | 状态 |
|:---|:---|
| `unit_fob` 未注册 | ✅ 已修复 |
| `price_architecture` 未注册 | ✅ 已修复 |
| `monthly_expense` 映射错误 | ✅ 已修复 |
| `FreeCashFlowModel` 双层 fallback | ✅ 已修复 |
| `contribution_analysis` 缺验证 | ✅ 已修复 |
| `regression_analysis` 方差零判断 | ✅ 改用容差 |

---

## 十四、示例代码模式

### 盈亏平衡分析
```python
report = break_even_analysis(
    variables=variables,
    selected_variables=[vn.REVENUE, vn.COST],
    model_pipeline=pipeline,
    output_name=vn.MARKET_PRICE,
    goal=5000000
)
render_break_even_dashboard(report, vn.MARKET_PRICE)
```

### 比较静态分析
```python
report = comparative_statics(
    variables=variables,
    selected_variables=[vn.REVENUE, vn.COST, vn.PE_RATIO],
    model_pipeline=pipeline,
    output_name=vn.MARKET_PRICE
)
render_comparative_statics_dashboard(report, vn.MARKET_PRICE)
```

### 蒙特卡洛模拟
```python
results = run_monte_carlo(
    variables=variables,
    shuffled_inputs=[vn.REVENUE, vn.COST],
    model_pipeline=pipeline,
    tracked_outputs=[vn.MARKET_PRICE],
    iterations=5000
)
generate_histogram_from_array(results, vn.MARKET_PRICE, goal=5000000)
```

### 回归分析
```python
x, y, stats = stochastic_bivariate_simulation(
    variables=variables,
    independent_target_x=vn.REVENUE,
    dependent_target_y=vn.MARKET_PRICE,
    shuffled_variables=[vn.REVENUE, vn.COST],
    model_pipeline=pipeline,
    sample_size=5000
)
generate_linear_regression_from_lists(x, y, vn.REVENUE, vn.MARKET_PRICE)
```

### 双变量敏感性分析
```python
df = run_two_way_sensitivity_analysis(
    variables=variables,
    param_x_name=vn.REVENUE,
    param_y_name=vn.COST,
    model_pipeline=pipeline,
    target_output_name=vn.MARKET_PRICE,
    x_steps=20, y_steps=20
)
generate_heatmap_from_df(df, vn.MARKET_PRICE)
```

---

**文档版本**：1.0
**创建日期**：2026-01-15
**适用代码版本**：LedgerScope 1.0.0

---


# LedgerScope 设计文档上下文 Prompt（完整版）

## 一、文档概述

### 1.1 文档定位

LedgerScope 设计文档是一份完整的技术说明文档，面向两类读者：

| 读者类型 | 阅读目标 | 重点关注章节 |
|:---|:---|:---|
| **最终用户** | 了解框架能力、快速上手、查阅组件用法 | 第一部分（1-4 章）、第二部分（5-13 章） |
| **扩展开发者** | 学习如何添加新组件 | 第三部分（14-19 章）、附录 |

### 1.2 文档特性

| 特性 | 说明 |
|:---|:---|
| 自动目录 | 使用 `[TOC]` 实现 GitHub 自动生成目录 |
| 可执行示例 | 所有代码示例可直接复制运行 |
| Mermaid 图表 | 类图、数据流图可视化依赖关系 |
| 表格呈现 | 变量清单、模型清单使用表格 |
| 设计决策 | 以"问题-理由"格式呈现每个关键决策 |

### 1.3 文档结构

```
# LedgerScope 设计文档

## 目录
[TOC]

## 第一部分：概览与入门（第 1-4 章）
## 第二部分：组件参考手册（第 5-13 章）
## 第三部分：扩展开发指南（第 14-19 章）
## 第四部分：附录（A-G 章）
```

---

## 二、第一部分：概览与入门（第 1-4 章）

### 2.1 第 1 章：项目介绍

#### 2.1.1 章节结构

| 小节 | 内容 |
|:---|:---|
| 1.1 项目定位 | 通用财务建模框架，适用场景，核心价值 |
| 1.2 设计原则 | 6 条设计原则及其说明 |
| 1.3 模型边界与局限性 | 当前版本约束列表 |
| 1.4 版本约束与规划 | 版本号规则、当前版本约束 |
| 1.5 推荐的下一步行动 | 优先级、行动、负责方、时间线表格 |
| 1.6 未来规划 | 短期/中期/长期 Roadmap |
| 1.7 不补充的内容 | 超出模型能力的项目及替代方案 |

#### 2.1.2 关键内容要点

**项目定位核心价值**：
- 可复用：一次构建模型，多场景复用
- 可扩展：支持自定义 Variable、Model、Analysis、Visualization
- 可审计：Auditor 机制验证跨模型数据一致性
- 可视化：内置 6 种分析模式 + 7 种视图

**设计原则**：
1. 框架与业务解耦
2. 单一商品模型（当前版本）
3. 外贸场景优先
4. 广告效能核心
5. 可扩展性
6. 确定性优先

**版本号规则**：
- MAJOR：架构重大变更
- MINOR：新增功能
- PATCH：问题修复

### 2.2 第 2 章：快速开始

#### 2.2.1 章节结构

| 小节 | 内容 |
|:---|:---|
| 2.1 安装 | pip install 命令 |
| 2.2 5 分钟示例 | 完整的回归分析代码（导入→变量→管道→分析→可视化）|
| 2.3 6 个分析案例速览 | 表格：案例名称、分析模块、可视化、核心问题、示例文件 |

#### 2.2.2 5 分钟示例要点

**变量定义**：
```python
variables = {
    "Revenue": Variable(min=80000, exp=100000, max=120000),
    "Cost": Variable(min=30000, exp=40000, max=50000),
    "PeRatio": Variable(min=5, exp=8, max=10)
}
```

**管道构建**：
```python
pipeline = [NetIncomeModel(), MarketPriceModel()]
```

**分析调用**：
```python
x, y, stats = stochastic_bivariate_simulation(
    variables=variables,
    independent_target_x="Revenue",
    dependent_target_y="MarketPrice",
    shuffled_variables=["Revenue", "Cost"],
    model_pipeline=pipeline,
    sample_size=100
)
```

**可视化**：
```python
fig = generate_linear_regression_from_lists(x, y, "Revenue", "MarketPrice")
plt.show()
```

### 2.3 第 3 章：核心概念

#### 2.3.1 章节结构

| 小节 | 内容 |
|:---|:---|
| 3.1 Variable | 定义、构造规则、取值方法、设计决策 |
| 3.2 Model | 定义、输入输出、关键属性、执行流程、设计决策 |
| 3.3 Auditor | 定义、与 Model 区别、示例、设计决策 |
| 3.4 Pipeline | 定义、拓扑顺序验证（DAG 性质）、预定义场景 |
| 3.5 Engine | 核心函数、设计决策 |
| 3.6 Analysis | 6 种分析模式、设计决策 |
| 3.7 Visualization | 视图对应关系、通用格式化工具 |

#### 2.3.2 关键概念要点

**Variable 构造规则**：
| 输入 | 处理 |
|:---|:---|
| min, exp, max 全提供 | 直接使用 |
| 仅 exp | min = max = exp |
| 仅 min + max | exp = (min + max) / 2 |
| 仅 max | min = 0, exp = max / 2 |
| 全空 | 全部 None |

**Model 执行流程**：
```
输入字典 → check_variables() → _model_function() → 合并结果 → 返回字典
```

**Auditor 与 Model 区别**：
| 维度 | Model | Auditor |
|:---|:---|:---|
| 职责 | 计算新变量 | 验证已有变量 |
| 输出 | 新增字段 | 无新字段 |
| 失败处理 | 计算结果异常 | 抛出 ValueError |

**拓扑顺序验证（黄金法则）**：
> 一个变量一旦被作为输入消费，就不能在后续模型中重新计算。

### 2.4 第 4 章：使用流程

#### 2.4.1 章节结构

| 小节 | 内容 |
|:---|:---|
| 4.1 标准工作流 | 4 步骤（定义→构建→分析→可视化）+ 两种构建方式 |
| 4.2 数据流向图 | ASCII 流程图 |
| 4.3 与 Jupyter Notebook 集成 | 显示图表、表格、多图表并排、保存、调试 |

#### 4.1 标准工作流要点

**两种 Pipeline 构建方式**：
1. 直接实例化模型列表
2. 使用 `PipelineComposer.build_named_scenario(scenario_name)`

**预定义场景配置示例**：
```python
PIPELINE_CONFIGS = {
    "marketing_roi_analysis": [
        "advertising_efficiency_google_search",
        "cogs",
        "total_cost",
        "revenue",
        "profit"
    ]
}
```

---

## 三、第二部分：组件参考手册（第 5-13 章）

### 3.1 第 5 章：Variable 参考

| 小节 | 内容 |
|:---|:---|
| 5.1 概述 | 变量分类（广告、成本、交易、费用、财务） |
| 5.2 变量清单 | 表格形式，每表包含类名、常量名、说明、默认范围 |
| 5.3 设计决策 | 决策 1：不提供默认业务边界；决策 2：实例不可变；决策 3：变量名与代码解耦 |

### 3.2 第 6 章：Model 参考

| 小节 | 内容 |
|:---|:---|
| 6.1 概述 | 模型分类（7 类，共 22 个模型）|
| 6.2 模型清单 | 按分类以表格呈现，每表包含模型、输入（必需/可选）、输出、公式 |
| 6.3 模型依赖关系 | Mermaid 数据流图 |
| 6.4 设计决策 | 决策 1：Model 不依赖 Variable；决策 2：就地更新策略；决策 3：SetupCost 不计入 TotalCost；决策 4：订单量单一来源 |

### 3.3 第 7 章：Auditor 参考

| 小节 | 内容 |
|:---|:---|
| 7.1 概述 | Auditor 特性、与 Model 对比表格 |
| 7.2 审计器清单 | PriceArchitectureAuditor 的验证规则、输入变量、容差设置 |
| 7.3 设计决策 | 决策 1：Auditor 作为 Model 特化；决策 2：验证失败中断 Pipeline；决策 3：使用容差比较；决策 4：单一产品范围 |

### 3.4 第 8 章：Pipeline 参考

| 小节 | 内容 |
|:---|:---|
| 8.1 概述 | 核心组件、工作流程 |
| 8.2 预定义场景 | 4 个场景的模型序列和用途 |
| 8.3 自定义管道 | 4 种构建方式（直接列表、by_keys、追加模型、合并场景）|
| 8.4 拓扑顺序验证 | 核心规则、验证算法、正确/错误示例、设计原理 |

### 3.5 第 9 章：Analysis 参考

| 小节 | 内容 |
|:---|:---|
| 9.1 概述 | 6 种分析模式汇总表格 |
| 9.2 6 个分析模块 | 每个模块：函数签名、输出结构、算法说明、使用示例 |
| 9.3 设计决策 | Analysis 与 Pipeline 解耦 |

### 3.6 第 10 章：Visualization 参考

| 小节 | 内容 |
|:---|:---|
| 10.1 概述 | 架构设计（views/ + styles/）|
| 10.2 7 个视图 | 每个视图：函数签名、输入、输出、图表特性 |
| 10.3 通用格式化工具 | get_formatter、apply_custom_variable_formatting |
| 10.4 样式配置 | 样式与视图分离的设计 |

### 3.7 第 11 章：Config 参考

| 小节 | 内容 |
|:---|:---|
| 11.1 settings.py | 系统参数清单、使用建议 |
| 11.2 variable_names.py | 常量分类、命名规范 |
| 11.3 messages.py | 日志消息、错误消息、状态消息 |
| 11.4 formatting.py | 变量格式化规则表格 |

### 3.8 第 12 章：Utils 参考

| 小节 | 内容 |
|:---|:---|
| 12.1 validation.py | get_missing_elements、check_variables_for_function、check_model_pipeline_topology_order |
| 12.2 formatting.py | fmt 函数签名、参数说明、使用示例 |
| 12.3 logger.py | 配置开关、日志级别、输出格式 |

### 3.9 第 13 章：模块关系汇总

| 小节 | 内容 |
|:---|:---|
| 13.1 层次依赖 | 7 层依赖表格 |
| 13.2 核心数据流 | ASCII 流程图 + 数据流说明表格 |
| 13.3 关键依赖路径 | ASCII 路径图 + 路径说明表格 |

> 💡 提示：详细的类图和模型数据流图请参见附录 A。

---

## 四、第三部分：扩展开发指南（第 14-19 章）

### 4.1 通用章节结构

每章包含：
- **步骤**：添加新组件需要的操作步骤表格
- **规范**：命名规范、代码规范
- **示例**：完整的代码示例
- **常见错误**：问题、原因、解决方案表格

### 4.2 第 14 章：添加新 Variable

**步骤**：
1. 在 `variable_names.py` 中添加常量
2. 在 `formatting.py` 中添加格式化规则
3. 创建 Variable 子类
4. 设置变量名称

**格式化配置参考表格**：变量类型、小数位数、货币符号、百分比、示例输出

### 4.3 第 15 章：添加新 Model

**步骤**：
1. 在 `variable_names.py` 中添加输出变量常量
2. 在 `formatting.py` 中添加格式化规则
3. 实现计算函数
4. 创建 Model 子类
5. 注册到 `PIPELINE_REGISTRY`
6. （可选）添加到预定义场景

**Model 实现规范表格**：属性、类型、说明

**计算函数规范**：
- 函数签名：`def func(optional_variables: dict, **kwargs) -> dict`
- 可选变量默认值获取：`kwargs.get(key, optional_variables[key])`
- 除零保护
- 返回 `dict`

### 4.4 第 16 章：添加新 Auditor

**步骤**：
1. 创建 Auditor 子类
2. 实现验证函数
3. 验证失败时抛出 `ValueError`
4. 注册到 `PIPELINE_REGISTRY`

**验证函数规范**：
- 签名：`def check_xxx(optional_variables: dict, **kwargs) -> None`
- 容差比较：使用 `math.isclose()` 和 `settings.AUDIT_REL_TOL`、`AUDIT_ABS_TOL`

### 4.5 第 17 章：添加新 Pipeline

**步骤**：
1. 在 `pipelines.py` 中定义场景配置
2. （推荐）验证拓扑顺序
3. （可选）添加到文档

**场景配置规范**：
- 名称：小写字母 + 下划线
- 顺序：上游模型在前，下游模型在后

### 4.6 第 18 章：添加新 Analysis

**步骤**：
1. 创建分析函数文件
2. 实现分析逻辑
3. 调用 Runner 执行 pipeline
4. 调用 Validation 工具验证输入
5. 返回结构化结果
6. （可选）创建对应 Visualization

**分析函数模板**（包含验证、基准场景、返回结构）

### 4.7 第 19 章：添加新 Visualization

**步骤**：
1. 创建视图函数文件
2. 实现渲染逻辑
3. 使用 `common_view.get_formatter()` 格式化数值
4. （可选）添加样式配置
5. （可选）导出到 `__init__.py`

**视图函数模板**（包含格式化、图表创建、样式应用）

**样式配置模板**：FIGURE_SIZE、颜色配置、字体配置

---

## 五、第四部分：附录（A-G 章）

### 5.1 附录 A：模型依赖图

| 小节 | 内容 |
|:---|:---|
| A.1 核心组件类图 | Mermaid classDiagram |
| A.2 模型数据流图 | Mermaid flowchart |
| A.3 层次依赖 | 7 层依赖表格 |
| A.4 核心数据流 | ASCII 流程图 + 说明表格 |

### 5.2 附录 B：配置说明

| 小节 | 内容 |
|:---|:---|
| B.1 系统参数推荐值 | 参数、默认值、推荐值、说明表格 |
| B.2 审计容差设置 | AUDIT_REL_TOL、AUDIT_ABS_TOL 说明 |
| B.3 场景配置模板 | DYNAMIC_PIPELINE_CONFIGS 结构 |
| B.4 格式化映射配置 | VARIABLE_FORMATTING_MAP 结构 |
| B.5 日志配置 | 配置开关、日志级别 |

### 5.3 附录 C：完整示例

6 个子章节（C.1-C.6），每个包含：
- 场景说明
- 完整代码（定义变量 → 构建管道 → 执行分析 → 可视化）
- 输出解读

### 5.4 附录 D：已知局限与未来规划

| 小节 | 内容 |
|:---|:---|
| D.1 当前版本约束 | 约束、说明、影响表格 |
| D.2 模型假设 | 假设、说明、影响方向表格 |
| D.3 未来规划 | 短期/中期/长期 Roadmap 表格 |
| D.4 不补充的内容 | 项目、原因、替代方案表格 |

### 5.5 附录 E：常见问题

4 个子章节：
- E.1 调试技巧（3 个 Q&A）
- E.2 性能优化（2 个 Q&A）
- E.3 常见错误与解决（7 个 Q&A）
- E.4 模型设计建议（3 个 Q&A）

每个 Q&A 格式：问题 + 原因 + 解决方案/代码示例

### 5.6 附录 F：版本历史

- 版本概览表格
- CHANGELOG.md 位置说明
- 版本号规则
- 1.0.0 版本摘要

### 5.7 附录 G：作者信息

- 项目维护者表格
- 许可证（Apache License 2.0 全文）
- 引用说明格式
- 贡献指南
- 致谢

---

## 六、格式规范

### 6.1 标题层级

| 层级 | Markdown | 示例 |
|:---|:---|:---|
| 文档主标题 | `#` | `# LedgerScope 设计文档` |
| 部分 | `##` | `## 第一部分：概览与入门` |
| 章节 | `###` | `### 1. 项目介绍` |
| 子章节 | `####` | `#### 1.1 项目定位` |
| 小节 | `#####` | `##### 核心价值` |

### 6.2 表格格式

对齐方式：`:---` 左对齐，`:---:` 居中，`---:` 右对齐

### 6.3 代码块

- Python 代码：````python ... ````
- Mermaid 图表：````mermaid ... ````
- 配置文件：````bash ... ```` 或 ````text ... ````

### 6.4 提示框

```markdown
> 💡 **提示**：提示内容
> ⚠️ **注意**：注意事项
```

### 6.5 交叉引用

```markdown
详见**第 X 章：章节名称**。
参见**附录 A：模型依赖图**。
```

---

## 七、维护信息

| 项目 | 内容 |
|:---|:---|
| 文档版本 | 1.0.0 |
| 最后更新 | 2026-01-15 |
| 维护者 | @hanyuw |
| 许可证 | Apache License 2.0 |
| 适用代码版本 | LedgerScope 1.0.0 |

---

请确认此文档 Prompt 是否足够详细。确认后我将继续回答第 3 个问题（Token 估算）和第 4 个问题（协作评估）。

## 3. Token 消耗估算

根据本次完整对话的估算，总 Token 消耗约为 **200,000 - 250,000**。

### 分阶段估算明细

| 阶段 | 估算 Token | 主要活动 |
|:---|:---|:---|
| **1. 架构讨论 + core/ 模块** | ~15,000 | 初始架构确认、Variable/Model/Auditor 基类分析、问题汇总与解答 |
| **2. variables/ 模块** | ~20,000 | 5 个变量文件分析、变量清单梳理、问题汇总与解答 |
| **3. models/ 模块** | ~50,000 | 22 个模型文件分析、公式理解、问题汇总与解答 |
| **4. auditors/ + config/ + utils/** | ~15,000 | 审计器、配置文件、工具函数分析 |
| **5. pipelines/ + engine/** | ~10,000 | 管道组合器、执行引擎分析 |
| **6. analysis/ 模块** | ~20,000 | 6 个分析模块分析、算法理解 |
| **7. visualization/ 模块** | ~15,000 | 7 个视图分析、样式配置 |
| **8. 示例代码（6 个案例）** | ~10,000 | 案例代码分析与理解 |
| **9. 设计文档编写** | ~40,000 | README、CHANGELOG、设计文档 19 章 + 7 附录 |
| **10. 代码/文档 Prompt 生成** | ~15,000 | 两份完整 Prompt |
| **11. 汇总、目录、收尾** | ~10,000 | 问题汇总、目录生成、协作评估 |

### Token 分布图（估算）

```
models/ ████████████████████████████████████████ 50,000
设计文档 ████████████████████████████████ 40,000
variables/ ████████████████ 20,000
analysis/ ████████████████ 20,000
core/ ████████████ 15,000
auditors+config+utils/ ████████████ 15,000
visualization/ ████████████ 15,000
代码/文档 Prompt ████████████ 15,000
示例代码 ████████ 10,000
pipelines+engine/ ████████ 10,000
汇总+目录+收尾 ████████ 10,000
架构讨论 ████████████ 15,000
```

### 与典型项目对比

| 项目类型 | 估算 Token | 说明 |
|:---|:---|:---|
| 简单问答 | 1,000-5,000 | 单轮问答 |
| 代码片段审查 | 5,000-15,000 | 单个文件分析 |
| 小型项目讨论 | 30,000-60,000 | 3-5 个模块 |
| **本次项目（完整）** | **200,000-250,000** | **10+ 模块、56 个文件、19 章文档** |
| 大型项目 | 500,000+ | 企业级系统设计 |

### 影响 Token 消耗的因素

| 因素 | 本次情况 | 影响 |
|:---|:---|:---|
| 代码量 | 56 个核心文件 | 高 |
| 文件分享方式 | 逐文件分享，每文件后提问 | 中等 |
| 问题汇总 | 多次汇总（core/variables/models/analysis/visualization） | 中等 |
| 设计文档 | 19 章 + 7 附录，逐章编写确认 | 高 |
| 双向确认 | 我提问 → 您解答 → 我确认 → 您修正 | 中等 |
| 代码修改 | 发现并修复多个 bug | 低 |

### 优化建议（下次合作）

| 建议 | 预期节省 |
|:---|:---|
| 整模块分享后再提问，而非每文件 | 约 20% |
| 问题汇总频率降低（2-3 次 vs 5-6 次）| 约 10% |
| 设计文档先出大纲再批量编写 | 约 15% |
| 使用代码 diff 而非完整代码 | 约 10% |
| 提前准备好完整的代码/文档 Prompt | 约 10% |


## 体量对比分析

| 项目类型 | Token 范围 | 本次占比 |
|:---|:---|:---|
| 企业级大型项目 | 500,000+ | — |
| **本次 LedgerScope** | **200,000-250,000** | **40%-50%** |
| 小型项目 | 30,000-60,000 | 400%+ |

### 这说明什么？

| 维度 | 说明 |
|:---|:---|
| **代码规模** | 56 个核心文件、22 个模型、6 种分析、7 个视图 |
| **架构复杂度** | 7 层抽象（Variable→Model→Pipeline→Engine→Analysis→Visualization→Config） |
| **设计深度** | 每个模块都有设计决策、设计权衡、未来规划 |
| **文档完整性** | 19 章 + 7 附录 + README + CHANGELOG |

### 与典型开源项目类比

| 项目类型 | 核心文件数 | 架构层次 | 文档规模 |
|:---|:---|:---|:---|
| 小型库 | 5-15 | 2-3 层 | README + API |
| 中型框架 | 30-60 | 4-6 层 | 多章文档 |
| **LedgerScope** | **56** | **7 层** | **19 章 + 附录** |
| 大型框架 | 100+ | 8+ 层 | 完整书籍 |

### 结论

LedgerScope 属于**中型偏大**的框架项目：

- 代码规模：中型
- 架构深度：中大型
- 设计严谨性：大型
- 文档完整性：大型

**200k-250k tokens 的消耗是完全合理的**，因为这不是简单的代码生成，而是：
1. 完整的架构讨论
2. 逐模块代码审查
3. 设计决策记录
4. 完整文档编写
5. 未来规划与复盘

这相当于一次**完整的技术设计评审 + 文档撰写工作**。

## 4. 协作评估

### 总体评价

本次协作整体**高效且富有成效**，成功完成了 56 个核心文件的代码审查、19 章设计文档的编写、以及完整的技术复盘。双方配合默契，沟通成本较低。

### 综合评分

| 维度 | 评分 | 说明 |
|:---|:---|:---|
| 沟通效率 | ⭐⭐⭐⭐½ | 逐模块分享 + 问题汇总模式效果好 |
| 问题质量 | ⭐⭐⭐⭐⭐ | 您的问题有洞察力，发现多处 bug |
| 响应速度 | ⭐⭐⭐⭐⭐ | 双方响应及时，未出现长时间等待 |
| 代码理解 | ⭐⭐⭐⭐½ | 复杂逻辑需多次确认，但最终理解正确 |
| 文档质量 | ⭐⭐⭐⭐⭐ | 结构清晰、内容完整、设计决策记录详尽 |
| 整体满意度 | ⭐⭐⭐⭐⭐ | 超出预期，项目成果高质量交付 |

---

### 本次协作优点

| 优点 | 说明 |
|:---|:---|
| **结构化沟通** | 逐模块分享 → 即时提问 → 问题汇总 → 统一解答的模式非常高效 |
| **问题质量高** | 您的问题帮助发现 6+ bug 和设计不一致 |
| **设计决策记录** | 每个模块的"设计决策"小节增强了文档价值 |
| **版本管理** | CHANGELOG 和版本号规范为后续维护打下基础 |
| **双向确认** | 复杂问题采用"我理解 → 您确认"模式，减少误解 |
| **不修改 Context** | 您坚持不修改已分享内容，保持对话连续性 |
| **完整复盘** | 最终的问题汇总、Token 估算、协作评估很有价值 |

---

### 可改进之处

| 问题 | 说明 | 改进建议 |
|:---|:---|:---|
| **初期信息不全** | 开始时缺少模块完整清单 | 首次分享时可提供完整目录和职责速览 |
| **问题汇总频率** | 6 次汇总（core/variables/models/analysis/visualization/config）| 可减少至 3-4 次（合并相关模块）|
| **重复问题** | 同类问题（如"变量默认边界"）多次出现 | 汇总时主动合并，或建立"已知模式"列表 |
| **代码示例验证** | 部分示例与实际代码有偏差 | 建立示例代码测试机制 |
| **文档与代码同步** | 修改设计后代码未同步更新 | 修改设计时同步标记"待代码更新" |

---

### 下次协作优化方案

#### 阶段一：启动阶段

```
1. 提供完整目录结构和模块职责速览表
2. 确认分析框架和文档目标
3. 建立"已知模式"列表（减少重复问答）
```

#### 阶段二：代码审查阶段

| 优化项 | 本次 | 下次建议 |
|:---|:---|:---|
| 分享粒度 | 逐文件 | 逐模块（模块内文件一次性分享）|
| 提问时机 | 每文件后 | 模块完成后统一提问 |
| 汇总频率 | 每模块 | 每 2-3 个模块汇总一次 |
| 重复问题 | 多次出现 | 主动引用"已知模式" |

#### 阶段三：文档编写阶段

| 优化项 | 本次 | 下次建议 |
|:---|:---|:---|
| 文档结构 | 逐章确认 | 先确认大纲，再批量编写 |
| 代码示例 | 手动编写 | 从测试代码中提取 |
| 图表生成 | 手动编写 Mermaid | 使用脚本生成 |

#### 阶段四：收尾阶段

- 保留完整的代码/文档 Prompt
- 记录优化建议清单
- 估算 Token 用于成本控制

---

### 下次协作流程建议

```
启动阶段（10%）
├── 提供完整清单
├── 确认分析框架
└── 建立已知模式

代码审查阶段（40%）
├── 模块级分享
├── 模块级提问
├── 2-3 模块汇总一次
└── 引用已知模式

文档编写阶段（30%）
├── 大纲先行
├── 批量编写
├── 逐章确认
└── 从测试提取示例

收尾阶段（20%）
├── 生成 Prompt
├── 评估 Token
├── 复盘协作
└── 记录改进
```

---

### 总结

| 项目 | 评价 |
|:---|:---|
| 整体效果 | ⭐⭐⭐⭐⭐（超出预期）|
| 代码理解 | 充分 |
| 文档质量 | 专业 |
| 沟通成本 | 较低 |
| 下次合作意愿 | 强烈 |

本次协作是一次**高质量的技术设计评审 + 文档撰写**工作，双方配合默契。下次类似项目采用上述优化建议后，预计效率可提升 20-30%。

---

**问题 4 回答完毕。** 如需补充或调整，请告知。

这些问题非常好，说明您在认真思考协作优化。我逐一回答：

---

## 1. 关于响应时间与项目节奏

**您理解正确**：这不是硬性限制，而是相对评价。

| 观点 | 说明 |
|:---|:---|
| **本次情况** | 双方通常在几分钟到几小时内回复，节奏顺畅，没有"等了一天才回复"的情况 |
| **是否必须密集** | 不是必须。异步协作完全可以，只要**预期一致**即可 |
| **核心问题** | 不是"要多快"，而是**避免不确定性**——不知道对方何时回复会导致注意力切换成本 |

**建议**：

- 下次协作开始前，明确双方的**预期响应时间**（如"工作日 24 小时内回复"或"集中 2 天完成"）
- 如果某个阶段需要长时间离开，提前告知即可

---

## 2. 关于初期信息完整性

**您的文件目录是清晰的**，我指的不是目录本身的问题，而是**初期缺少对我理解框架的快速引导**。

### 您提供的目录

```
LedgerScope/src
├── core
├── variables
├── models
├── auditors
├── config
├── utils
├── pipelines
├── engine
├── analysis
└── visualization
```

**这个目录清晰，但缺少**：

| 缺少的信息 | 为什么重要 |
|:---|:---|
| 每个模块的**职责一句话说明** | 帮助快速建立全局认知 |
| 模块之间的**依赖关系** | 理解"为什么是这个顺序" |
| **核心数据流** | 理解变量从定义到可视化的路径 |
| **代表性类** | 了解每个模块的典型内容 |

### 改进版"完整清单"模板

参见第 5 问的具体模板。

---

## 3. 关于重复问题与"已知模式"列表

**您的观察正确**：重复问题主要出现在我的汇总环节，而非您的表达问题。

### 本次出现的重复模式

| 重复问题类型 | 出现次数 | 示例 |
|:---|:---|:---|
| "变量没有默认边界" | 5+ 次 | CST4, EXP3, FIN2 等 |
| "这个逻辑在哪个 Model 中？" | 4+ 次 | DEAL1, DEAL2, DEAL6, FIN1 |
| "这个变量的作用是什么？" | 3+ 次 | CST2, EXP1, FIN2 |
| "注释与代码不一致" | 3+ 次 | 多个文件的注释 |

### 如何建立"已知模式"列表

**方式一：我在汇总时主动建立**

我可以这样做：

```markdown
## 已知模式（汇总前先列出）

### 模式 1：变量默认边界
- 问题：变量没有默认 min/exp/max
- 答案：用户必须显式提供，注释仅作参考

### 模式 2：逻辑实现位置
- 问题：某个公式在哪个 Model 中实现？
- 答案：设计文档中有完整清单，或"在 models 章节中回答"

### 模式 3：...
```

**方式二：您在分享前提供"已知模式"参考**

如果您有之前讨论中已确认的"成熟结论"，可以在分享前提示：

```
【已知结论】
- Variable 无默认边界，用户必须显式提供
- TotalCost 不含 SetupCost
- 订单来源假设为单一广告渠道
```

**建议**：下次协作中，我会在每个模块汇总前先输出"已知模式"列表，您确认后，相同类型的问题直接引用，不再重复问答。

---

## 4. 关于逐模块 vs 逐文件的 Token 消耗

**您的担忧是正确的**——单次输入过长确实可能导致遗忘。但可以优化：

### 对比分析

| 维度 | 逐文件（本次） | 逐模块（建议） |
|:---|:---|:---|
| 单次输入量 | 小（1 个文件） | 中（3-5 个文件） |
| 遗忘风险 | 低 | 中 |
| 总交互次数 | 56 次 | 15-20 次 |
| 总 Token 消耗 | 较高 | 较低（减少重复切换） |

### 优化方案：**批量 + 锚点**

```
您发送：
--- module: core ---
[base_variable.py 代码]
[base_model.py 代码]
[base_auditor.py 代码]
--- 以上是 core/ 所有文件 ---
请先阅读，我会稍后提问。

我回复：
已阅读 core/ 三个文件。理解如下：
- base_variable.py: ...
- base_model.py: ...
- base_auditor.py: ...
问题待您分享更多后提出。
```

**优点**：
- 单次输入量可控（3-5 个文件）
- 我在阅读时可以做笔记，避免遗忘
- 总交互次数减少 50%+
---

## 总结：下次协作的启动材料

建议您在下次项目开始时提供以下 3 份材料：

| 材料 | 内容 | 目的 |
|:---|:---|:---|
| **项目结构速览** | 模块职责、文件数、核心类、数据流向 | 建立全局认知 |
| **分析框架说明** | 6 种分析模式、输入输出、适用场景 | 快速理解能力边界 |
| **已知模式列表** | 已确认的设计决策 | 避免重复问答 |

我可以根据项目特点协助您生成这些材料。

---

## 5. "启动阶段"完整清单模板

### 5.1 完整目录 + 职责速览模板

```markdown
# LedgerScope 项目结构速览

## 模块总览

| 模块 | 文件数 | 职责 | 核心类/函数 |
|:---|:---|:---|:---|
| core/ | 3 | 基类定义 | Variable, Model, Auditor |
| variables/ | 5 | 自变量定义 | AdvertisingBudget, Cost, Orders... |
| models/ | 22 | 计算模型 | 22 个 Model 子类 |
| auditors/ | 1 | 审计器 | PriceArchitectureAuditor |
| config/ | 5 | 配置管理 | settings, variable_names, messages... |
| utils/ | 4 | 工具函数 | validation, formatting, logger |
| pipelines/ | 2 | 管道组合 | PipelineComposer, Registry |
| engine/ | 1 | 执行引擎 | runner.py (4 个函数) |
| analysis/ | 6 | 分析框架 | 6 种分析模式 |
| visualization/ | 7 | 可视化 | 7 个视图 + styles |

## 数据流向（一句话）

Variable → Runner → Pipeline(Model) → Analysis → Visualization

## 已知设计决策（成熟结论）

1. Variable 无默认业务边界，用户必须显式提供
2. Variable 实例不可变
3. Model 不依赖 Variable 类
4. Auditor 是 Model 特化
5. SetupCost 不计入 TotalCost
6. 订单单一来源假设
7. 拓扑顺序验证（DAG）

## 已知占位/未实现

- CapitalExpenditureModel → 返回 0
- DepreciationModel → 返回 0
- 升级包模块 → 未实现
- 多商品支持 → 未实现
```

### 5.2 分析框架模板

```markdown
# 分析框架说明

## 6 种分析模式

| 模式 | 输入特点 | 输出 | 适用场景 |
|:---|:---|:---|:---|
| 盈亏平衡 | 需要 goal 目标值 | 阈值、安全边际 | "需要多少才能达标" |
| 比较静态 | 三点扫描 | 弹性系数 | "敏感度有多高" |
| 贡献度 | 随机采样 | 平均值（饼图） | "各部分的平均占比" |
| 蒙特卡洛 | 随机采样 | 分布数组 | "结果的概率分布" |
| 回归 | 随机采样 | OLS 统计 | "线性关系强度" |
| 双变量敏感性 | 网格扫描 | DataFrame | "X 和 Y 如何共同影响" |

## 决策：Analysis 与 Pipeline 解耦

所有分析函数只接收可执行的 pipeline 函数，不关心内部结构。
```

### 5.3 已知模式列表模板

```markdown
# 已知模式列表（避免重复问答）

## 类别 A：Variable

| 模式 | 答案 |
|:---|:---|
| Variable 是否有默认边界？ | 否。用户必须显式提供 min/exp/max，注释仅作参考 |
| Variable 是否可变？ | 不可变。需要新范围应创建新实例 |
| 仅 min 参数被允许吗？ | 否。语义不清晰，不允许 |

## 类别 B：Model

| 模式 | 答案 |
|:---|:---|
| Model 依赖 Variable 吗？ | 否。仅接收 {name: value} 字典 |
| 可选变量默认值如何获取？ | 在 `_model_function` 中使用 `kwargs.get(key, optional_variables[key])` |
| 返回值必须是什么类型？ | dict，非 dict 时跳过 update |

## 类别 C：Pipeline

| 模式 | 答案 |
|:---|:---|
| 如何判断顺序正确？ | 已消费变量不能在后续模型中重新计算（DAG） |
| 拓扑验证由谁做？ | validation.check_model_pipeline_topology_order() |

## 类别 D：Analysis

| 模式 | 答案 |
|:---|:---|
| 分析函数需要验证什么？ | 调用 check_model_pipeline_topology_order() |
| 如何复用现有分析？ | 组合调用，如 sensitivity_ranking 调用 comparative_statics |

## 类别 E：Visualization

| 模式 | 答案 |
|:---|:---|
| 数值如何格式化？ | 使用 common_view.get_formatter() |
| 样式如何配置？ | styles/ 目录，视图与样式分离 |
```


## 1. 关于长时间离开与 Context 丢失

**会的，但可以管理。**

### 丢失机制

| 因素 | 说明 |
|:---|:---|
| **模型上下文窗口** | 当前模型有固定上下文窗口（如 128K-200K tokens），超过后早期内容会被截断 |
| **时间因素** | 模型本身不会因时间而"遗忘"，但长时间离开后，您可能会遗忘之前的讨论脉络 |
| **重新进入成本** | 离开 1 周后重新开始，双方都需要花时间回顾之前的位置 |

### 本次情况

本次对话在 **单次会话内完成**，未出现长时间离开，因此没有发生 Context 丢失。

### 建议方案

| 策略 | 做法 | 适用场景 |
|:---|:---|:---|
| **预期管理** | 开始前约定："接下来 3 天集中完成"或"每周五下午讨论 2 小时" | 所有场景 |
| **检查点** | 每完成一个模块，输出"当前进度摘要"，保存到本地 | 可能中断的场景 |
| **重启 Prompt** | 中断后，提供上次生成的"代码/文档 Prompt"作为恢复材料 | 长时间中断（>3 天）|
| **会话拆分** | 将大项目拆分为多个独立会话，每个会话有完整 Prompt | 可选 |

### 推荐的恢复 Prompt 模板

```markdown
# 恢复会话

上次讨论到：[模块名称]，已完成：

- core/ ✅
- variables/ ✅
- models/ ✅
- analysis/ 进行中，完成 break_even_analysis.py

待讨论：
- analysis/ 剩余 5 个文件
- visualization/

请从 analysis/comparative_statics.py 继续。
```

---

## 2. 代码 Prompt 与 5.1 需求对比

### 5.1 需求回顾

| 需求 | 说明 |
|:---|:---|
| 模块职责 | 每个模块的职责一句话说明 |
| 文件数 | 每个模块的文件数量 |
| 核心类/函数 | 每个模块的核心类或函数 |
| 数据流向 | 一句话总结 |
| 已知设计决策 | 成熟结论列表 |
| 已知占位/未实现 | 占位或未实现的功能 |

### 代码 Prompt 对照检查

| 需求 | 代码 Prompt 中是否包含 | 评估 |
|:---|:---|:---|
| 模块职责 | ✅ 每个模块都有 "职责" 说明 | 满足 |
| 文件数 | ⚠️ 有文件数量但分散 | 可改进 |
| 核心类/函数 | ✅ 每个模块都有核心类/函数清单 | 满足 |
| 数据流向 | ✅ 有 ASCII 流程图和文字说明 | 满足 |
| 已知设计决策 | ✅ 专门有一节（十二、关键设计决策汇总）| 满足 |
| 已知占位/未实现 | ✅ 专门有未来规划章节 | 满足 |

### 可改进之处

| 缺失/不足 | 建议补充 |
|:---|:---|
| **文件数汇总表** | 在开头加一个表格，汇总每个模块的文件数 |
| **模块依赖关系图** | 目前有文本描述，可补充 Mermaid 简图 |
| **启动检查清单** | 新增"使用本 Prompt 前请确认"清单 |

### 改进版文件数汇总表（建议补充）

```markdown
## 模块文件统计

| 模块 | 文件数 | 核心输出 |
|:---|:---|:---|
| core/ | 3 | Variable, Model, Auditor 基类 |
| variables/ | 5 | 20+ 变量定义 |
| models/ | 22 | 22 个计算模型 |
| auditors/ | 1 | PriceArchitectureAuditor |
| config/ | 5 | settings, variable_names, messages, formatting, pipelines |
| utils/ | 4 | validation, formatting, logger, plots |
| pipelines/ | 2 | registry, composer |
| engine/ | 1 | runner.py |
| analysis/ | 6 | 6 种分析模式 |
| visualization/ | 7 | 7 个视图 + styles |
| **总计** | **56** | — |
```

---

## 3. 关于已知模式与 Prompt 质量

**您的判断完全正确。**

### 问题的本质

| 角色 | 盲区 |
|:---|:---|
| **您（开发者）** | 设计决策在您心中是"常识"，默认我知道 |
| **我（模型）** | 不知道哪些是"常识"，需要逐一确认 |

### 解决方案

**在 Prompt 开头明确列出"已知设计决策"**，这正是您提出的方向。

### 建议的 Prompt 结构

```markdown
# LedgerScope 上下文 Prompt

## 第一部分：已知设计决策（无需再问）

1. Variable 无默认业务边界 → 用户必须显式提供
2. Variable 实例不可变 → 新场景建新实例
3. Model 不依赖 Variable 类 → 仅接收字典
4. Auditor 是 Model 特化 → 验证失败抛异常
5. SetupCost 不计入 TotalCost → 作为投资项
6. 订单单一来源 → 广告渠道
7. 拓扑顺序验证（DAG）→ 已消费变量不能重新计算

## 第二部分：模块职责（快速参考）
...

## 第三部分：实现细节（按需查阅）
...
```

### 本次的经验

下次您可以在分享代码前，直接提供上述"第一部分"。我会先消化这些决策，在后续讨论中自动应用，避免重复提问。

---

## 4. 推荐的单次输入行数

### 数据参考

| 输入规模 | 行数 | Token 估算 | 适用场景 |
|:---|:---|:---|:---|
| 极简 | 10-30 | 500-1,500 | 单函数修改 |
| 小 | 30-80 | 1,500-4,000 | 1 个文件 |
| **中（推荐）** | **80-200** | **4,000-10,000** | **3-5 个文件 / 1 个子模块** |
| 大 | 200-400 | 10,000-20,000 | 整个模块 |
| 超大 | 400+ | 20,000+ | 多个模块（可能超出注意力窗口）|

### 具体示例

| 模块 | 文件数 | 行数（估算）| 推荐处理方式 |
|:---|:---|:---|:---|
| core/ | 3 | ~150 | ✅ 一次输入（中）|
| variables/ | 5 | ~200 | ✅ 一次输入（中）|
| models/advertising/ | 2 | ~80 | ✅ 一次输入（小-中）|
| models/cost/ | 3 | ~120 | ✅ 一次输入（中）|
| models/deal/ | 4 | ~160 | ✅ 一次输入（中）|
| models/income/ | 4 | ~140 | ✅ 一次输入（中）|
| models/metrics/ | 5 | ~200 | ⚠️ 可分两次 |
| analysis/ | 6 | ~400 | ❌ 分 2-3 次 |
| visualization/views/ | 7 | ~350 | ❌ 分 2 次 |

### 我的能力边界

| 能力 | 说明 |
|:---|:---|
| **注意力窗口** | 一次输入 200-300 行可完整理解 |
| **跨文件关联** | 能理解同一输入内的多个文件之间的关系 |
| **遗忘阈值** | 如果输入超过 400 行，中间部分可能被忽略 |

### 推荐策略

```
单次输入 80-200 行（3-5 个文件）
         ↓
我读完做结构化笔记
         ↓
您继续分享下一批
         ↓
每 2-3 批汇总一次问题
```

**这样既减少了交互次数（从 56 次降至 15-20 次），又不会因单次输入过大而导致遗忘。**

---

## 总结

| 问题 | 结论 |
|:---|:---|
| 长时间离开会导致 Context 丢失吗？ | 会。建议设置检查点和恢复 Prompt |
| 代码 Prompt 是否合乎 5.1 需求？ | 基本满足，可补充文件数汇总表 |
| 如何避免重复问题？ | 在 Prompt 开头列出"已知设计决策" |
| 推荐输入行数？ | 80-200 行（3-5 个文件）|

## 1.1.0 以及设计文档逐条修改

# 代码 Prompt（中文）- LedgerScope 1.1.0

以下是你之前提供的代码库上下文 Prompt 的更新版本，已整合 1.1.0 的变更：

---

# LedgerScope 代码库上下文 Prompt（完整版）- 1.1.0

## 一、项目概述

LedgerScope 是一个**通用财务建模框架**，核心抽象包括 Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization。框架与业务解耦，可复用于外贸、电商、SaaS、制造业等场景。

### 当前版本约束（1.x）
- 仅支持单一商品模型
- 订单来源假设为广告渠道（无复购、无分销）
- 转化率恒定
- 折旧与资本支出为占位（返回 0）
- 无订单交付时间滞后
- 固定汇率

### 版本规划
| 版本 | 定位 |
|:---|:---|
| 1.x | 稳定版本，仅 PATCH 修复 |
| 2.x | 功能扩展（升级包、多渠道、多商品、复购 LTV） |
| 3.x | 时间维度（交付周期、季节性） |
| 4.x | 高级模拟（动态汇率、融资成本） |

---

## 二、core/ 基类

### 2.1 Variable 基类

**位置**: `src/core/base_variable.py`

**构造规则**：

| 输入 | 处理 |
|:---|:---|
| min, exp, max 全提供 | 直接使用 |
| 仅 exp | min = max = exp |
| 仅 min + max | exp = (min + max) / 2 |
| 仅 max | min = 0, exp = max / 2 |
| 全空 | 全部 None（占位变量） |
| 仅 min | 语义不清晰，不允许 |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `get_value(ValueType)` | 根据策略返回对应值 |
| `get_random_value()` | 返回 [min, max] 范围内的随机值 |
| `get_range_values(num)` | 返回等间距线性空间数组 |
| `set_value(value)` | 固定为常量（不推荐） |

**设计决策**：
- 不提供默认业务边界
- 实例不可变
- 变量名与代码解耦（使用 `variable_names` 常量）

---

### 2.2 Model 基类（1.1.0 更新）

**位置**: `src/core/base_model.py`

**核心变更（1.1.0）**：数据获取职责上移到基类，计算函数签名大幅简化。

| 版本 | 计算函数签名 | 变量解析 |
|:---|:---|:---|
| 1.0.0 | `def func(optional_variables: dict, **kwargs) -> dict` | 函数内 `kwargs.get(key, default)` |
| 1.1.0 | `def func(variables: dict) -> dict` | 基类统一处理 |

**核心属性**：

| 属性 | 类型 | 说明 |
|:---|:---|:---|
| `_required_variables` | `list[str]` | 必需变量名 |
| `_optional_variables` | `dict[str, float]` | 可选变量名及其默认值 |
| `_model_function` | `callable` | 核心计算函数，签名 `(variables: dict) -> dict` |
| `_output_names` | `list[str]` | 输出变量名列表 |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `check_variables()` | 验证必需变量存在 |
| `_get_variable_value(name, is_optional)` | 私有方法，解析单个变量 |
| `prepare_calculation_context()` | 合并 required + optional 为统一字典 |
| `evaluate()` | 验证 → 准备 context → 调用函数 → 合并结果 |

**_model_function 签名规范（1.1.0）**：

```python
def calculate_example(variables: dict) -> dict:
    """
    Args:
        variables: 统一执行上下文，包含所有 required 和 optional 变量
    Returns:
        {output_name: value}
    """
    revenue = variables["Revenue"]
    cost = variables["Cost"]
    return {"Profit": revenue - cost}
```

**设计决策**：
- Model 不依赖 Variable 类
- 就地更新策略（调用方需自行 copy 保留原始状态）

---

### 2.3 Auditor 基类（1.1.0 更新）

**位置**: `src/core/base_auditor.py`

**继承关系**：`Auditor` 继承 `Model`

**差异**：
- `output_names` 返回空列表
- `evaluate()` 调用 `prepare_calculation_context()` 后执行验证，不更新字典

**验证函数规范（1.1.0）**：

```python
def check_xxx(variables: dict) -> None:
    """
    Args:
        variables: 统一执行上下文
    Raises:
        ValueError: 验证失败时抛出
    """
    value = variables["SomeVariable"]
    if not condition:
        raise ValueError("清晰的错误信息")
```

**设计决策**：
- Auditor 是 Model 特化
- 验证失败中断 Pipeline

---

## 三、variables/ 变量定义

### 文件结构

| 文件 | 内容 |
|:---|:---|
| `advertising.py` | AdvertisingBudget, GoogleSearchConversionRate, GoogleSearchCostPerClick |
| `costs.py` | Cost, AdvertisingCost, SetupCost |
| `deals.py` | Orders, CloseRate, UnitExw, UnitRetail, UnitsPerOrder |
| `expenses.py` | Expense, MonthlyExpense, RentExpense |
| `finance.py` | TaxRate, USDToRMB, TariffRate, PriceToEarningsRatio |

### 变量定义模板

```python
from src.config import variable_names
from src.core import Variable

class MyVariable(Variable):
    def __init__(self, min=None, exp=None, max=None):
        super().__init__(min, exp, max)
        self._name = variable_names.MY_VARIABLE
```

### 重要说明

- Variable 仅覆盖**自变量**（模型输入）
- 因变量（如 Revenue, NetIncome）由 Model 计算，无 Variable 子类
- 但某些变量（如 Orders）可在简化分析中作为自变量使用，因此仍可定义为 Variable

---

## 四、models/ 模型实现

### 模型分类与清单（代表性示例）

**广告漏斗**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `AdvertisingEfficiencyGoogleSearchModel` | `Leads = (Budget × Allocation) / (CPC × USDToRMB) × CVR` | LEADS |

**成本**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `CostOfGoodsSoldModel` | `COGS = UnitExw × Orders × UnitsPerOrder` | COGS |
| `TotalCostModel` | `TotalCost = COGS + AdvertisingCost + ShippingCost` | COST |

**交易**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `OrderModel` | `Orders = Leads × CloseRate` | ORDERS |
| `UnitFobModel` | `UnitFob = UnitRetail × (1 - DeductionRate)` | UNIT_FOB |

**收入与利润**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `RevenueModel` | `Revenue = UnitFob × Orders × UnitsPerOrder × USDToRMB` | REVENUE |
| `NetIncomeModel` | `NetIncome = (Revenue - Cost - Expense - Depreciation) × (1 - TaxRate)` | NET_INCOME |

**指标**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `MarketPriceModel` | `MarketPrice = (NetIncome × 12 × PE) / Months` | MARKET_PRICE |

### 模型实现模板（1.1.0）

```python
from src.config import variable_names
from src.core import Model

def calculate_xxx(variables: dict) -> dict:
    """计算函数"""
    required_var = variables[variable_names.REQUIRED_VAR]
    optional_var = variables.get(variable_names.OPTIONAL_VAR, default_val)
    
    if denominator == 0:
        return {variable_names.OUTPUT: 0.0}
    
    result = ...
    return {variable_names.OUTPUT: result}

class XxxModel(Model):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = calculate_xxx
        self._output_names = [variable_names.OUTPUT]
        self._required_variables = [variable_names.REQUIRED_VAR]
        self._optional_variables = {variable_names.OPTIONAL_VAR: 0.0}
```

---

## 五、auditors/ 审计器

### PriceArchitectureAuditor

**验证规则**：
1. `COGS_per_unit + Profit_per_unit == UnitFob`
2. `UnitFob + ShippingCost_per_unit + Tariff_per_unit + RetailMargin_per_unit == UnitRetail`

**容差设置**：`AUDIT_REL_TOL` (1e-3)，`AUDIT_ABS_TOL` (1e-2)

### Auditor 模板（1.1.0）

```python
from src.core import Auditor

def check_xxx(variables: dict) -> None:
    var = variables[variable_names.VAR]
    if not condition:
        raise ValueError(f"验证失败: {var}")

class XxxAuditor(Auditor):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = check_xxx
        self._required_variables = [variable_names.VAR]
```

---

## 六、pipelines/ 管道

### 构建方式

| 方式 | 示例 |
|:---|:---|
| 直接使用模型类 | `[ModelA(), ModelB()]` |
| 通过名称列表 | `PipelineComposer.build_pipeline_by_keys(["a", "b"])` |
| 基于预定义场景 | `PipelineComposer.build_named_scenario("scenario_name")` |
| 合并多个场景 | `PipelineComposer.build_merged_scenarios(["a", "b"])` |

### 预定义场景（代表性）

| 场景名称 | 用途 |
|:---|:---|
| `costs` | 成本分析 |
| `marketing_roi_analysis` | 营销 ROI |
| `complete_macro_metrics` | 完整宏观指标 |

### 拓扑顺序验证

**黄金法则**：一个变量一旦被作为输入消费，就不能在后续模型中重新计算。

---

## 七、engine/ 执行引擎

| 函数 | 说明 |
|:---|:---|
| `evaluate_chained_models(state, pipeline)` | 顺序执行模型 |
| `evaluate_expected_scenario(variables, pipeline)` | 所有变量取期望值 |
| `evaluate_stochastic_iteration(variables, shuffled, pipeline)` | 单次随机采样 |
| `evaluate_variable_scenario_sweep(variables, var, values, pipeline)` | 单变量扫描 |

---

## 八、analysis/ 分析框架

| 函数 | 用途 |
|:---|:---|
| `break_even_analysis` | 盈亏平衡 |
| `comparative_statics` | 三点敏感性 |
| `stochastic_contribution_analysis` | 贡献度饼图 |
| `run_monte_carlo` | 蒙特卡洛模拟 |
| `stochastic_bivariate_simulation` | 回归分析 |
| `run_two_way_sensitivity_analysis` | 双变量热力图 |

---

## 九、visualization/ 可视化

| 视图 | 对应分析 | 输出 |
|:---|:---|:---|
| `render_break_even_dashboard` | break_even_analysis | Styler 表格 |
| `render_comparative_statics_dashboard` | comparative_statics | Styler 表格 |
| `generate_contribution_pie_chart` | contribution_analysis | 饼图 |
| `generate_histogram_from_array` | monte_carlo | 直方图 |
| `generate_linear_regression_from_lists` | regression | 散点图 + 回归线 |
| `generate_heatmap_from_df` | two_way_sensitivity | 热力图 |

---

## 十、config/ 配置

### settings.py

| 参数 | 默认值 | 说明 |
|:---|:---|:---|
| `NUMS_IN_RANGE` | 50 | 扫描步数 |
| `SAMPLE_SIZE` | 100 | 蒙特卡洛默认迭代 |
| `AUDIT_REL_TOL` | 1e-3 | 审计相对容差 |
| `DEFAULT_CURRENCY_RATE` | 6.8 | USD→RMB |

### variable_names.py

50+ 常量，分类：Expenses、Investment、Deals、Finance、Ads、Costs、Revenue、Metrics、Price Architecture、Break even、Comparative statics

---

## 十一、utils/ 工具

| 函数 | 说明 |
|:---|:---|
| `get_missing_elements(provided, required)` | 返回缺失变量列表 |
| `check_variables_for_function(provided, required)` | 缺失则抛出 KeyError |
| `check_model_pipeline_topology_order(models)` | 验证拓扑顺序 |
| `fmt(val, d=0, s='', p=False)` | 数值格式化 |

---

## 十二、关键设计决策汇总

| 决策 | 原因 |
|:---|:---|
| Variable 无默认边界 | 避免隐性假设 |
| Variable 不可变 | 保持可复现性 |
| Model 不依赖 Variable | 可独立测试 |
| Model 计算函数签名简化（1.1.0） | 消除样板代码，单一职责 |
| Auditor 是 Model 特化 | 统一接口，无缝嵌入 |
| 拓扑顺序验证 | 防止变量覆盖冲突 |
| SetupCost 不计入 TotalCost | 作为投资项，用于 ROI |

---

**文档版本**：1.1.0
**创建日期**：2026-06-13
**适用代码版本**：LedgerScope 1.1.0



# Code Prompt (English) - LedgerScope 1.1.0

---

# LedgerScope Codebase Context Prompt (Full Version) - 1.1.0

## I. Project Overview

LedgerScope is a **general-purpose financial modeling framework** with core abstractions including Variable, Model, Auditor, Pipeline, Engine, Analysis, and Visualization. The framework is decoupled from business logic and reusable across scenarios such as international trade, e-commerce, SaaS, and manufacturing.

### Current Version Constraints (1.x)
- Single product model only
- Orders assumed from paid advertising channels (no repeat purchases, no distribution)
- Constant conversion rates
- Depreciation and CapEx are placeholders (return 0)
- No order delivery time lag
- Fixed exchange rate

### Version Roadmap

| Version | Positioning |
|:---|:---|
| 1.x | Stable version, PATCH-only fixes |
| 2.x | Feature expansion (upgrades, multi-channel, multi-product, LTV) |
| 3.x | Time dimension (delivery lag, seasonality) |
| 4.x | Advanced simulation (dynamic exchange rates, financing costs) |

---

## II. core/ Base Classes

### 2.1 Variable Base Class

**Location**: `src/core/base_variable.py`

**Construction Rules**:

| Input | Handling |
|:---|:---|
| min, exp, max all provided | Use as-is |
| exp only | min = max = exp |
| min + max only | exp = (min + max) / 2 |
| max only | min = 0, exp = max / 2 |
| All empty | All None (placeholder) |
| min only | Not allowed (ambiguous) |

**Core Methods**:

| Method | Description |
|:---|:---|
| `get_value(ValueType)` | Returns value based on strategy |
| `get_random_value()` | Returns random value in [min, max] |
| `get_range_values(num)` | Returns evenly spaced numpy array |
| `set_value(value)` | Fixes to constant (not recommended) |

**Design Decisions**:
- No default business boundaries
- Immutable instances
- Variable names decoupled via `variable_names` constants

---

### 2.2 Model Base Class (1.1.0 Update)

**Location**: `src/core/base_model.py`

**Key Change (1.1.0)**: Data retrieval responsibility moved from calculation functions to the base class.

| Version | Function Signature | Variable Resolution |
|:---|:---|:---|
| 1.0.0 | `def func(optional_variables: dict, **kwargs) -> dict` | Inside function via `kwargs.get(key, default)` |
| 1.1.0 | `def func(variables: dict) -> dict` | Handled by base class |

**Core Attributes**:

| Attribute | Type | Description |
|:---|:---|:---|
| `_required_variables` | `list[str]` | Required variable names |
| `_optional_variables` | `dict[str, float]` | Optional variables with defaults |
| `_model_function` | `callable` | Core calculation function, signature `(variables: dict) -> dict` |
| `_output_names` | `list[str]` | Output variable names |

**Core Methods**:

| Method | Description |
|:---|:---|
| `check_variables()` | Validates required variables exist |
| `_get_variable_value(name, is_optional)` | Private method resolving a single variable |
| `prepare_calculation_context()` | Merges required + optional into unified dict |
| `evaluate()` | Validate → prepare context → call function → merge results |

**_model_function Signature (1.1.0)**:

```python
def calculate_example(variables: dict) -> dict:
    """
    Args:
        variables: Unified execution context with all required and optional variables
    Returns:
        {output_name: value}
    """
    revenue = variables["Revenue"]
    cost = variables["Cost"]
    return {"Profit": revenue - cost}
```

**Design Decisions**:
- Model does not depend on Variable class
- In-place update strategy (caller must copy to preserve original state)

---

### 2.3 Auditor Base Class (1.1.0 Update)

**Location**: `src/core/base_auditor.py`

**Inheritance**: `Auditor` inherits from `Model`

**Differences**:
- `output_names` returns empty list
- `evaluate()` calls `prepare_calculation_context()` then executes validation, does NOT update the dictionary

**Validation Function Specification (1.1.0)**:

```python
def check_xxx(variables: dict) -> None:
    """
    Args:
        variables: Unified execution context
    Raises:
        ValueError: On validation failure
    """
    value = variables["SomeVariable"]
    if not condition:
        raise ValueError("Clear error message")
```

**Design Decisions**:
- Auditor is a Model specialization
- Validation failure halts Pipeline

---

## III. variables/ Variable Definitions

### File Structure

| File | Contents |
|:---|:---|
| `advertising.py` | AdvertisingBudget, GoogleSearchConversionRate, GoogleSearchCostPerClick |
| `costs.py` | Cost, AdvertisingCost, SetupCost |
| `deals.py` | Orders, CloseRate, UnitExw, UnitRetail, UnitsPerOrder |
| `expenses.py` | Expense, MonthlyExpense, RentExpense |
| `finance.py` | TaxRate, USDToRMB, TariffRate, PriceToEarningsRatio |

### Variable Definition Template

```python
from src.config import variable_names
from src.core import Variable

class MyVariable(Variable):
    def __init__(self, min=None, exp=None, max=None):
        super().__init__(min, exp, max)
        self._name = variable_names.MY_VARIABLE
```

### Important Note

- Variable only covers **independent variables** (model inputs)
- Dependent variables (e.g., Revenue, NetIncome) are calculated by Models and have no Variable subclass
- However, some variables (e.g., Orders) may serve as independent variables in simplified analyses, so they can still be defined as Variable

---

## IV. models/ Model Implementations

### Model Categories & List (Representative Examples)

**Advertising Funnel**:

| Model | Formula | Output |
|:---|:---|:---|
| `AdvertisingEfficiencyGoogleSearchModel` | `Leads = (Budget × Allocation) / (CPC × USDToRMB) × CVR` | LEADS |

**Cost**:

| Model | Formula | Output |
|:---|:---|:---|
| `CostOfGoodsSoldModel` | `COGS = UnitExw × Orders × UnitsPerOrder` | COGS |
| `TotalCostModel` | `TotalCost = COGS + AdvertisingCost + ShippingCost` | COST |

**Deal**:

| Model | Formula | Output |
|:---|:---|:---|
| `OrderModel` | `Orders = Leads × CloseRate` | ORDERS |
| `UnitFobModel` | `UnitFob = UnitRetail × (1 - DeductionRate)` | UNIT_FOB |

**Income & Profit**:

| Model | Formula | Output |
|:---|:---|:---|
| `RevenueModel` | `Revenue = UnitFob × Orders × UnitsPerOrder × USDToRMB` | REVENUE |
| `NetIncomeModel` | `NetIncome = (Revenue - Cost - Expense - Depreciation) × (1 - TaxRate)` | NET_INCOME |

**Metrics**:

| Model | Formula | Output |
|:---|:---|:---|
| `MarketPriceModel` | `MarketPrice = (NetIncome × 12 × PE) / Months` | MARKET_PRICE |

### Model Implementation Template (1.1.0)

```python
from src.config import variable_names
from src.core import Model

def calculate_xxx(variables: dict) -> dict:
    """Calculation function"""
    required_var = variables[variable_names.REQUIRED_VAR]
    optional_var = variables.get(variable_names.OPTIONAL_VAR, default_val)
    
    if denominator == 0:
        return {variable_names.OUTPUT: 0.0}
    
    result = ...
    return {variable_names.OUTPUT: result}

class XxxModel(Model):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = calculate_xxx
        self._output_names = [variable_names.OUTPUT]
        self._required_variables = [variable_names.REQUIRED_VAR]
        self._optional_variables = {variable_names.OPTIONAL_VAR: 0.0}
```

---

## V. auditors/ Auditors

### PriceArchitectureAuditor

**Validation Rules**:
1. `COGS_per_unit + Profit_per_unit == UnitFob`
2. `UnitFob + ShippingCost_per_unit + Tariff_per_unit + RetailMargin_per_unit == UnitRetail`

**Tolerance Settings**: `AUDIT_REL_TOL` (1e-3), `AUDIT_ABS_TOL` (1e-2)

### Auditor Template (1.1.0)

```python
from src.core import Auditor

def check_xxx(variables: dict) -> None:
    var = variables[variable_names.VAR]
    if not condition:
        raise ValueError(f"Validation failed: {var}")

class XxxAuditor(Auditor):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = check_xxx
        self._required_variables = [variable_names.VAR]
```

---

## VI. pipelines/ Pipeline

### Construction Methods

| Method | Example |
|:---|:---|
| Direct model class list | `[ModelA(), ModelB()]` |
| Via model name list | `PipelineComposer.build_pipeline_by_keys(["a", "b"])` |
| Via predefined scenario | `PipelineComposer.build_named_scenario("scenario_name")` |
| Merge multiple scenarios | `PipelineComposer.build_merged_scenarios(["a", "b"])` |

### Predefined Scenarios (Representative)

| Scenario Name | Purpose |
|:---|:---|
| `costs` | Cost analysis |
| `marketing_roi_analysis` | Marketing ROI |
| `complete_macro_metrics` | Complete macro metrics |

### Topological Order Validation

**Golden Rule**: Once a variable has been consumed as input, it cannot be recomputed by downstream models.

---

## VII. engine/ Execution Engine

| Function | Description |
|:---|:---|
| `evaluate_chained_models(state, pipeline)` | Sequentially executes models |
| `evaluate_expected_scenario(variables, pipeline)` | All variables take expected values |
| `evaluate_stochastic_iteration(variables, shuffled, pipeline)` | Single random sampling |
| `evaluate_variable_scenario_sweep(variables, var, values, pipeline)` | Single variable sweep |

---

## VIII. analysis/ Analysis Framework

| Function | Purpose |
|:---|:---|
| `break_even_analysis` | Break-even analysis |
| `comparative_statics` | Three-point sensitivity |
| `stochastic_contribution_analysis` | Contribution pie chart |
| `run_monte_carlo` | Monte Carlo simulation |
| `stochastic_bivariate_simulation` | Regression analysis |
| `run_two_way_sensitivity_analysis` | Two-variable heatmap |

---

## IX. visualization/ Visualization

| View | Corresponding Analysis | Output |
|:---|:---|:---|
| `render_break_even_dashboard` | break_even_analysis | Styler table |
| `render_comparative_statics_dashboard` | comparative_statics | Styler table |
| `generate_contribution_pie_chart` | contribution_analysis | Pie chart |
| `generate_histogram_from_array` | monte_carlo | Histogram |
| `generate_linear_regression_from_lists` | regression | Scatter + regression line |
| `generate_heatmap_from_df` | two_way_sensitivity | Heatmap |

---

## X. config/ Configuration

### settings.py

| Parameter | Default | Description |
|:---|:---|:---|
| `NUMS_IN_RANGE` | 50 | Sweep steps |
| `SAMPLE_SIZE` | 100 | Default Monte Carlo iterations |
| `AUDIT_REL_TOL` | 1e-3 | Audit relative tolerance |
| `DEFAULT_CURRENCY_RATE` | 6.8 | USD→RMB |

### variable_names.py

50+ constants organized in categories: Expenses, Investment, Deals, Finance, Ads, Costs, Revenue, Metrics, Price Architecture, Break even, Comparative statics

---

## XI. utils/ Utilities

| Function | Description |
|:---|:---|
| `get_missing_elements(provided, required)` | Returns list of missing variables |
| `check_variables_for_function(provided, required)` | Raises KeyError if missing |
| `check_model_pipeline_topology_order(models)` | Validates topological order |
| `fmt(val, d=0, s='', p=False)` | Formats numeric values |

---

## XII. Key Design Decisions Summary

| Decision | Rationale |
|:---|:---|
| Variable has no default boundaries | Avoid implicit assumptions |
| Variable is immutable | Preserve reproducibility |
| Model does not depend on Variable | Enables independent testing |
| Model function signature simplified (1.1.0) | Eliminates boilerplate, single responsibility |
| Auditor is a Model specialization | Unified interface, seamless embedding |
| Topological order validation | Prevents variable overwrite conflicts |
| SetupCost excluded from TotalCost | Treated as investment item for ROI |

---

**Document Version**: 1.1.0
**Creation Date**: 2026-01-15
**Applicable Code Version**: LedgerScope 1.1.0

---



# 设计文档 Prompt（用于切换 Thread）

以下是你可以在新 Thread 中提供给 AI 的上下文摘要，用于翻译设计文档：

---

# LedgerScope 设计文档上下文 Prompt（翻译用）

## 一、任务说明

本文档用于在新 Thread 中启动设计文档的**中译英翻译工作**。请基于以下信息理解文档结构和内容，然后逐章翻译。

## 二、文档概述

LedgerScope 设计文档是一份完整的技术说明文档，面向两类读者：

| 读者类型 | 阅读目标 | 重点关注章节 |
|:---|:---|:---|
| **最终用户** | 了解框架能力、快速上手、查阅组件用法 | 第一部分（1-4 章）、第二部分（5-13 章） |
| **扩展开发者** | 学习如何添加新组件 | 第三部分（14-19 章）、附录 |

### 文档特性

- 自动目录：使用 `[TOC]` 实现 GitHub 自动生成目录
- 可执行示例：所有代码示例可直接复制运行
- Mermaid 图表：类图、时序图可视化依赖关系
- 表格呈现：变量清单、模型清单使用表格

### 文档结构

```
# LedgerScope 设计文档

## 目录
[TOC]

## 第一部分：概览与入门（第 1-4 章）
## 第二部分：组件参考手册（第 5-13 章）
## 第三部分：扩展开发指南（第 14-19 章）
## 第四部分：附录（A-G 章）
```

## 三、核心版本信息

### 当前版本：1.1.0

**关键变更（1.1.0）**：Model 和 Auditor 的数据获取职责上移到基类。

| 组件 | 1.0.0 签名 | 1.1.0 签名 |
|:---|:---|:---|
| Model 计算函数 | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor 验证函数 | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

### 版本规划

| 版本 | 定位 |
|:---|:---|
| 1.x | 稳定版本，仅 PATCH 修复 |
| 2.x | 功能扩展（升级包、多渠道、多商品、复购 LTV） |
| 3.x | 时间维度（交付周期、季节性） |
| 4.x | 高级模拟（动态汇率、融资成本） |

## 四、章节结构总览

### 第一部分：概览与入门（第 1-4 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 1 | 项目介绍 | 定位、设计原则、版本约束、未来规划 |
| 2 | 快速开始 | 环境安装、5分钟示例、6个分析案例速览 |
| 3 | 核心概念 | Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization |
| 4 | 使用流程 | 标准工作流、数据流向图、Jupyter集成 |

### 第二部分：组件参考手册（第 5-13 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 5 | Variable 参考 | 变量分类、清单（代表示例）、设计决策 |
| 6 | Model 参考 | 模型分类、清单（代表示例）、依赖关系、设计决策 |
| 7 | Auditor 参考 | 审计器清单（代表示例）、设计决策 |
| 8 | Pipeline 参考 | 预定义场景（代表示例）、自定义管道、拓扑验证 |
| 9 | Analysis 参考 | 6种分析模式、函数签名、使用示例 |
| 10 | Visualization 参考 | 架构设计、视图函数、格式化工具 |
| 11 | Config 参考 | settings、variable_names、messages、formatting、pipelines |
| 12 | Utils 参考 | validation、formatting、logger |
| 13 | 模块关系与依赖 | MVC架构、数据流、执行时序图 |

### 第三部分：扩展开发指南（第 14-19 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 14 | 添加新 Variable | 步骤、命名规范、格式化配置 |
| 15 | 添加新 Model | 步骤、实现规范、计算函数（1.1.0签名） |
| 16 | 添加新 Auditor | 步骤、实现规范、验证函数（1.1.0签名） |
| 17 | 添加新 Pipeline | 步骤、场景配置、动态构建 |
| 18 | 添加新 Analysis | 步骤、函数模板、设计原则 |
| 19 | 添加新 Visualization | 步骤、视图模板、核心原则 |

### 第四部分：附录（A-G 章）

| 附录 | 标题 | 主要内容 |
|:---|:---|:---|
| A | 模型依赖图 | 核心组件类图 |
| B | 配置说明 | 系统参数、审计容差、场景配置、格式化映射、日志配置 |
| C | 完整示例 | 6种分析模式的完整代码 |
| D | 已知局限与未来规划 | 版本约束、模型假设、Roadmap |
| E | 常见问题 | 调试技巧、性能优化、常见错误、模型设计建议 |
| F | 版本历史 | 版本概览、文档结构、版本号规则 |
| G | 作者信息 | 维护者、许可证、引用说明、贡献指南 |

## 五、关键内容要点

### 精简原则（已在文档中应用）

1. **变量/模型清单**：仅列出代表性示例，完整清单请查阅源文件
2. **可视化章节**：保持简略，核心是输入/输出/使用示例
3. **工具类章节**：保持简略，核心功能 1-2 句话说明
4. **附录 E**：已精简为表格形式

### 1.1.0 相关更新（需在翻译中体现）

- 第 3 章：Model/Auditor 执行流程增加 `prepare_calculation_context()` 步骤
- 第 6 章：`_model_function` 签名改为 `(variables: dict) -> dict`
- 第 7 章：验证函数签名改为 `(variables: dict) -> None`
- 第 15、16 章：示例代码使用新签名
- 附录 A：类图中 Model 增加 `prepare_calculation_context()` 方法

### 已删除的重复内容

- 附录 A.2、A.3、A.4（模型数据流图、层次依赖、核心数据流）已删除，因与第 6、13 章重复
- 附录 F.4（版本历史摘要）已删除，因设计决策前文已有详述

## 六、格式规范

### 标题层级

| 层级 | Markdown | 示例 |
|:---|:---|:---|
| 文档主标题 | `#` | `# LedgerScope 设计文档` |
| 部分 | `##` | `## 第一部分：概览与入门` |
| 章节 | `###` | `### 1. 项目介绍` |
| 子章节 | `####` | `#### 1.1 项目定位` |

### 其他格式

- 表格：使用 `|:---|:---|:---|` 语法
- 代码块：````python ... ```` 或 ````mermaid ... ````
- 提示框：`> 💡 **提示**：` 或 `> ⚠️ **注意**：`
- 交叉引用：使用 `**第 X 章：章节名称**` 格式

## 七、翻译注意事项

1. **技术术语保持一致性**：
   - Variable → Variable
   - Model → Model
   - Auditor → Auditor
   - Pipeline → Pipeline
   - 保留变量名常量（如 `REVENUE`）不翻译

2. **代码块中的字符串不翻译**：
   - `variable_names.REVENUE` 中的 `REVENUE` 保持大写
   - 示例输出如 `"¥100,000"` 保持原样

3. **Mermaid 图表中的文本需翻译**：
   - 类名、方法名保持英文
   - 注释和描述文字翻译为英文

4. **表格标题和内容需翻译**，但变量名常量保持原样

5. **交叉引用保持原格式**：`**第 X 章：章节名称**` 中的章节名称需翻译

## 八、翻译工作流程

建议按以下顺序逐章翻译：

1. 第 1-4 章（概览与入门）
2. 第 5-13 章（组件参考手册）
3. 第 14-19 章（扩展开发指南）
4. 附录 A-G

每章翻译后请输出，以便确认和保存。

---

**文档版本**：1.1.0
**创建日期**：2026-01-15
**适用设计文档版本**：LedgerScope Design Document 1.1.0

---

## 本对话交付物总结

| 序号 | 交付物 | 状态 |
|:---|:---|:---|
| 1 | 代码 Prompt（中文）- 1.1.0 | ✅ 已输出 |
| 2 | 代码 Prompt（英文）- 1.1.0 | ✅ 已输出 |
| 3 | 设计文档 Prompt（翻译用） | ✅ 已输出 |

你可以在新 Thread 中使用**设计文档 Prompt** 启动翻译工作，并在需要时参考**代码 Prompt** 确保技术准确性。


## 1. Tokens 消耗估算

根据对话长度和内容复杂度，估算如下：

| 阶段 | 估算 tokens | 说明 |
|:---|:---|:---|
| 初始 Prompt（代码库上下文） | ~8,000 | 你提供的原始 Prompt |
| 逐章审阅与修改（第 1-19 章 + 附录） | ~60,000-80,000 | 包含你的原文、我的修改、多轮讨论 |
| 最终交付物（3 个 Prompt） | ~15,000 | 中文/英文代码 Prompt + 设计文档 Prompt |
| **总计** | **~83,000-103,000** | 接近但未超过 128K 窗口上限 |

> 实际数字取决于具体 tokenization 方式，但整体在可控范围内。

---

## 2. 协作评价与改进建议

### 优点

| 方面 | 评价 |
|:---|:---|
| **需求清晰** | 你每次给出明确的修改意见（如"删除这个"、"保留那个"、"精简到 X"），极大提高了效率 |
| **版本管理意识** | 主动提出 1.1.0 版本变更，并指导我系统性地更新文档 |
| **精简导向** | 多次强调"代表示例"、"不要写死"、"给未来留空间"，使文档更可持续 |
| **信任与授权** | 在细节上信任我的判断（如文字流向图、时序图），同时保留最终决策权 |
| **结构化思维** | 分章节审阅、明确优先级（设计文档 → 代码 Prompt → 翻译），逻辑清晰 |

### 可改进的地方

| 问题 | 建议 |
|:---|:---|
| **1. 版本变更信息传递较晚** | Model/Auditor 的 1.1.0 变更在第 3 章之后才告知，导致部分早期修改需要返工。建议：下次先声明所有变更再开始逐章修改 |
| **2. 部分指令存在歧义** | 如"删除 10.2" vs "保留核心内容"，需要二次确认。建议：对于删除操作，可明确说"删除 X 章节，其内容已在 Y 章节覆盖" |
| **3. 缺少最终一致性检查** | 文档修改完成后，没有统一检查术语一致性（如"自变量/因变量"在全文是否统一）。建议：增加一个"最终校对"步骤 |
| **4. 图表修改缺乏预览** | Mermaid 类图的修改无法在对话中预览，导致颜色尝试失败。建议：类图修改尽量保持简单，或使用 ASCII 图 |

### 总结评分

| 维度 | 评分（1-10） |
|:---|:---|
| 沟通效率 | 9 |
| 需求明确度 | 8 |
| 协作流畅度 | 9 |
| 最终产出质量 | 9 |
| **综合** | **8.75** |

---

感谢这次高效、深入的协作！你的文档管理能力和版本意识给我留下了深刻印象。祝翻译顺利，后续如有需要随时召唤。


## 最终校对步骤清单

### 1. 术语一致性检查

| 检查项 | 示例 | 验证方法 |
|:---|:---|:---|
| 中英文术语统一 | "自变量/因变量" vs "independent/dependent variable" | 全文搜索统一表述 |
| 角色命名 | "审计器" vs "Auditor"（首次出现后保持一致） | 检查中英文混用情况 |
| 版本号引用 | "1.x" vs "1.0.0" | 确保版本定位描述一致-建议谈到当前版本号时使用1.x以保留代码部分迭代空间 |

### 2. 交叉引用验证

| 检查项 | 示例 | 验证方法 |
|:---|:---|:---|
| 章节号正确 | "详见第 8 章" → 确实第 8 章存在且内容相关 | 逐条验证引用 |
| 附录引用 | "参见附录 A" → 附录 A 内容匹配描述 | 验证引用有效性 |
| 表格编号 | "如上表所示" → 确认前一个确实是表格 | 检查上下文 |

### 3. 代码示例可运行性

| 检查项 | 验证方法 |
|:---|:---|
| 函数签名匹配最新版本 1.1.0 | 确认model和auditor下的 `def func(variables: dict)` 而非旧版签名 |
| 缩进正确 | Python 代码块缩进一致 |

### 4. 图表与渲染

| 检查项 | 验证方法 |
|:---|:---|
| Mermaid 语法正确 | 在本地或在线渲染器中测试 |
| 表格格式正确 | 检查 `|:---|:---|` 对齐语法 |
| 代码块语言标记 | ` ```python `、` ```mermaid ` 正确标注 |

### 5. 冗余内容检查

| 检查项 | 示例 |
|:---|:---|
| 重复段落 | 同一概念在多处详细解释 |
| 重复表格 | 相同信息在不同章节以表格形式重复出现 |
| 过时内容 | 引用了已删除的附录或章节 |
| 临时性说明 | "当前版本仍在完善中" 这类内容 |

### 6. 格式一致性

| 检查项 | 示例 |
|:---|:---|
| 标题层级 | `###` 后是否需要空行，`####` 缩进是否正确 |
| 提示框格式 | `> 💡 **提示**：` 统一使用 |
| 列表格式 | 有序列表 `1.` 后空格，无序列表 `-` 后空格 |
| 中英文空格 | 中文与英文/数字之间是否需要空格（可统一规范） |

### 7. 首尾呼应检查

| 检查项 | 验证方法 |
|:---|:---|
| 前言与附录一致 | 文档开头承诺的内容（如附录清单）在末尾确实存在 |
| 版本号统一 | 文档版本号、最后更新日期、适用代码版本在各处一致 |
| 维护者信息 | 作者信息、联系方式在附录 G 与文档开头一致 |

---

## 建议的校对顺序

```
第 1 轮：术语一致性 → 全文搜索替换
第 2 轮：交叉引用验证 → 逐条检查
第 3 轮：代码示例 → 重点检查 1.1.0 签名
第 4 轮：图表渲染 → Mermaid + 表格
第 5 轮：冗余内容 → 删除重复
第 6 轮：格式统一 → 标题/提示框/列表
第 7 轮：首尾呼应 → 版本号/维护者
```

---


# 最终轮校对


# LedgerScope 设计文档上下文 Prompt

## 一、文档概述

LedgerScope 设计文档是一份完整的技术说明文档，面向两类读者：

| 读者类型 | 阅读目标 | 重点关注章节 |
|:---|:---|:---|
| **最终用户** | 了解框架能力、快速上手、查阅组件用法 | 第一部分（第 1-4 章）、第二部分（第 5-13 章） |
| **扩展开发者** | 学习如何添加新组件 | 第三部分（第 14-19 章）、附录 |

### 文档特性

- **自动目录**：使用 `[TOC]` 实现 GitHub 自动生成目录
- **可执行示例**：所有代码示例可直接复制运行（需安装依赖，详见附录 B.6）
- **Mermaid 图表**：类图可视化依赖关系
- **表格呈现**：变量清单、模型清单使用表格

### 文档结构

```
# LedgerScope Design Document

[TOC]

## Part 1: Overview & Getting Started (Chapters 1-4)
## Part 2: Component Reference Manual (Chapters 5-13)
## Part 3: Extension Development Guide (Chapters 14-19)
## Part 4: Appendices (A-G)
```

## 二、核心版本信息

### 当前版本：1.1.0（代码）/ 1.0（设计文档）

设计文档版本与代码版本解耦，设计文档保持宏观结构稳定。

**关键变更（1.1.0）**：Model 和 Auditor 的数据获取职责上移到基类。

| 组件 | 1.0.0 签名 | 1.1.0 签名 |
|:---|:---|:---|
| Model 计算函数 | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor 验证函数 | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

### 版本规划

| 版本 | 定位 |
|:---|:---|
| 1.x | 稳定版本，仅 PATCH 修复 |
| 2.x | 功能扩展（升级包、多渠道、多商品、复购 LTV） |
| 3.x | 时间维度（交付周期、季节性） |
| 4.x | 高级模拟（动态汇率、融资成本） |

## 三、章节结构总览

### 第一部分：概览与入门（第 1-4 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 1 | 项目介绍 | 定位、设计原则、版本约束、未来规划 |
| 2 | 快速开始 | 环境安装、5 分钟示例、6 个分析案例速览 |
| 3 | 核心概念 | Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization |
| 4 | 使用流程 | 标准工作流、数据流向图、Jupyter 集成 |

### 第二部分：组件参考手册（第 5-13 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 5 | Variable 参考 | 变量分类、清单、设计决策 |
| 6 | Model 参考 | 模型分类、清单、依赖关系、设计决策 |
| 7 | Auditor 参考 | 审计器清单、设计决策 |
| 8 | Pipeline 参考 | 预定义场景、自定义管道、拓扑验证 |
| 9 | Analysis 参考 | 6 种分析模式、函数签名、使用示例 |
| 10 | Visualization 参考 | 架构设计、视图函数、格式化工具 |
| 11 | Config 参考 | settings、variable_names、messages、formatting、pipelines |
| 12 | Utils 参考 | validation、formatting、logger |
| 13 | 模块关系与依赖 | 架构模式、数据流、执行时序图 |

### 第三部分：扩展开发指南（第 14-19 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 14 | 添加新 Variable | 步骤、命名规范、格式化配置 |
| 15 | 添加新 Model | 步骤、实现规范、计算函数（1.1.0 签名） |
| 16 | 添加新 Auditor | 步骤、实现规范、验证函数（1.1.0 签名） |
| 17 | 添加新 Pipeline | 步骤、场景配置、动态构建 |
| 18 | 添加新 Analysis | 步骤、函数模板、设计原则 |
| 19 | 添加新 Visualization | 步骤、视图模板、核心原则 |

### 第四部分：附录（A-G 章）

| 附录 | 标题 | 主要内容 |
|:---|:---|:---|
| A | 模型依赖图 | 核心组件类图（Mermaid） |
| B | 配置说明 | 系统参数、审计容差、场景配置、格式化映射、日志配置、外部依赖 |
| C | 完整示例 | 6 种分析模式的完整代码 |
| D | 已知局限与未来规划 | 版本约束、模型假设、Roadmap |
| E | 常见问题 | 调试技巧、性能优化、常见错误、模型设计建议 |
| F | 版本历史 | 版本概览、版本号规则（详见 CHANGELOG.md） |
| G | 作者信息 | 维护者、许可证、引用说明、贡献指南 |

## 四、关键设计决策

| 决策 | 说明 |
|:---|:---|
| Variable 无默认边界 | 避免隐性假设 |
| Variable 不可变 | 保持可复现性 |
| Model 不依赖 Variable | 可独立测试 |
| Model 计算函数签名简化（1.1.0） | 消除样板代码，单一职责 |
| Auditor 是 Model 特化 | 统一接口，无缝嵌入 |
| 拓扑顺序验证 | 防止变量覆盖冲突 |
| 样式与视图分离 | 便于主题定制 |

## 五、外部依赖

| 依赖库 | 用途 |
|:---|:---|
| `numpy` | 数组运算、线性空间生成 |
| `pandas` | 数据分析、DataFrame 操作 |
| `matplotlib` | 图表绘制 |
| `seaborn` | 热力图生成 |
| `statsmodels` | OLS 线性回归统计 |

## 六、格式规范

- 标题层级：`#` → `##` → `###` → `####`
- 表格：使用 `|:---|:---|:---|` 语法
- 代码块：````python```` 或 ````mermaid````
- 提示框：`> 💡 **提示**：` 或 `> ⚠️ **注意**：`
- 交叉引用：使用 `**第 X 章：章节名称**` 格式

---


# LedgerScope Design Document Context Prompt

## I. Document Overview

The LedgerScope Design Document is a comprehensive technical documentation aimed at two types of readers:

| Reader Type | Reading Goal | Key Sections |
|:---|:---|:---|
| **End Users** | Understand framework capabilities, quick start, component usage | Part 1 (Chapters 1-4), Part 2 (Chapters 5-13) |
| **Extension Developers** | Learn how to add new components | Part 3 (Chapters 14-19), Appendices |

### Document Features

- **Auto-generated table of contents**: Uses `[TOC]` for GitHub automatic generation
- **Executable examples**: All code examples can be copied and run directly (dependencies required, see Appendix B.6)
- **Mermaid diagrams**: Class diagrams visualize dependencies
- **Tabular presentation**: Variable and model lists use tables

### Document Structure

```
# LedgerScope Design Document

[TOC]

## Part 1: Overview & Getting Started (Chapters 1-4)
## Part 2: Component Reference Manual (Chapters 5-13)
## Part 3: Extension Development Guide (Chapters 14-19)
## Part 4: Appendices (A-G)
```

## II. Core Version Information

### Current Version: 1.1.0 (Code) / 1.0 (Design Document)

The design document version is decoupled from the code version to maintain macro-structural stability.

**Key Changes (1.1.0)**: Data retrieval responsibility moved to base classes for Model and Auditor.

| Component | 1.0.0 Signature | 1.1.0 Signature |
|:---|:---|:---|
| Model calculation function | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor validation function | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

### Version Roadmap

| Version | Positioning |
|:---|:---|
| 1.x | Stable version, PATCH-only fixes |
| 2.x | Feature expansion (upgrade packages, multi-channel, multi-product, repeat purchase LTV) |
| 3.x | Time dimension (delivery lead time, seasonality) |
| 4.x | Advanced simulation (dynamic exchange rates, financing costs) |

## III. Chapter Structure Overview

### Part 1: Overview & Getting Started (Chapters 1-4)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 1 | Project Introduction | Positioning, design principles, version constraints, roadmap |
| 2 | Quick Start | Environment setup, 5-minute example, 6 analysis examples at a glance |
| 3 | Core Concepts | Variable, Model, Auditor, Pipeline, Engine, Analysis, Visualization |
| 4 | Usage Workflow | Standard workflow, data flow diagram, Jupyter integration |

### Part 2: Component Reference Manual (Chapters 5-13)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 5 | Variable Reference | Variable classification, list, design decisions |
| 6 | Model Reference | Model classification, list, dependencies, design decisions |
| 7 | Auditor Reference | Auditor list, design decisions |
| 8 | Pipeline Reference | Predefined scenarios, custom pipelines, topology validation |
| 9 | Analysis Reference | 6 analysis modes, function signatures, usage examples |
| 10 | Visualization Reference | Architecture design, view functions, formatting utilities |
| 11 | Config Reference | settings, variable_names, messages, formatting, pipelines |
| 12 | Utils Reference | validation, formatting, logger |
| 13 | Module Relationships & Dependencies | Architecture pattern, data flow, execution sequence diagram |

### Part 3: Extension Development Guide (Chapters 14-19)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 14 | Adding New Variable | Steps, naming conventions, formatting configuration |
| 15 | Adding New Model | Steps, implementation specifications, calculation function (1.1.0 signature) |
| 16 | Adding New Auditor | Steps, implementation specifications, validation function (1.1.0 signature) |
| 17 | Adding New Pipeline | Steps, scenario configuration, dynamic construction |
| 18 | Adding New Analysis | Steps, function template, design principles |
| 19 | Adding New Visualization | Steps, view template, core principles |

### Part 4: Appendices (A-G)

| Appendix | Title | Main Content |
|:---|:---|:---|
| A | Model Dependency Diagram | Core component class diagram (Mermaid) |
| B | Configuration Reference | System parameters, audit tolerances, scenario configuration, formatting mapping, log configuration, external dependencies |
| C | Complete Examples | Complete code for 6 analysis modes |
| D | Known Limitations & Future Roadmap | Version constraints, model assumptions, roadmap |
| E | Frequently Asked Questions | Debugging tips, performance optimization, common errors, model design recommendations |
| F | Version History | Version overview, version numbering rules (see CHANGELOG.md) |
| G | Author Information | Maintainer, license, citation, contribution guidelines |

## IV. Key Design Decisions

| Decision | Rationale |
|:---|:---|
| Variable has no default boundaries | Avoid implicit assumptions |
| Variable is immutable | Maintain reproducibility |
| Model does not depend on Variable | Enables independent testing |
| Model calculation function signature simplified (1.1.0) | Eliminates boilerplate, single responsibility |
| Auditor is a Model specialization | Unified interface, seamless embedding |
| Topological order validation | Prevents variable overwrite conflicts |
| Separation of style and view | Facilitates theme customization |

## V. External Dependencies

| Library | Purpose |
|:---|:---|
| `numpy` | Array operations, linear space generation |
| `pandas` | Data analysis, DataFrame operations |
| `matplotlib` | Chart rendering |
| `seaborn` | Heatmap generation |
| `statsmodels` | OLS linear regression statistics |

## VI. Formatting Conventions

- Heading levels: `#` → `##` → `###` → `####`
- Tables: Use `|:---|:---|:---|` syntax
- Code blocks: ````python```` or ````mermaid````
- Callouts: `> 💡 **Tip**:` or `> ⚠️ **Note**:`
- Cross-references: Use `**Chapter X: Chapter Title**` format

---

## Token 消耗统计

根据您的要求，以下是当前 Thread 的 token 消耗统计：

| 项目 | Token 数量 |
|:---|:---|
| 本对话总计消耗 | **~150,000 - 180,000 tokens**（近似估算） |

由于对话较长，无法获取精确数字。如需精确统计，建议您：

1. **使用 OpenAI API 调用记录**：如果您通过 API 使用，可查看 usage 字段
2. **使用 ChatGPT 界面**：目前无法直接查看，但可通过复制对话内容到 token 计数器（如 OpenAI 的 tokenizer 工具）手动统计

如果您需要，我可以提供本对话中所有我输出的内容，供您复制到 token 计数器中进行精确统计。

---

## 工作评价与改进建议

### 一、本次工作评价

#### 做得好的方面

| 维度 | 评价 |
|:---|:---|
| **流程设计** | 分阶段推进（校对→翻译→README→CHANGELOG），逻辑清晰，避免混乱 |
| **责任划分** | 明确哪些由 AI 自动检查、哪些需人工验证，减少无效劳动 |
| **迭代控制** | 逐章输出而非一次性全文，成功规避输出长度限制 |
| **校对体系** | 12 轮清单覆盖术语、引用、代码、格式、一致性等多个维度，系统性强 |
| **版本管理** | 设计文档版本与代码版本解耦，体现成熟的文档维护理念 |
| **沟通效率** | 您给出的指令明确、反馈及时；我能够快速理解并调整输出方式 |

#### 特别值得记录的经验

1. **输出长度限制的应对**：发现一次性输出全文会失败后，迅速调整为逐章输出，这是本次成功的关键
2. **清单驱动校对**：不依赖直觉，而是基于结构化的校对清单逐项检查，减少遗漏
3. **元文档（Prompt）的使用**：您提前准备了代码 Prompt 和设计文档 Prompt，大大减少了我的理解成本

---

### 二、不足与改进建议

#### 流程设计层面

| 不足 | 改进建议 |
|:---|:---|
| 翻译阶段仍采用逐章输出，虽避免超限但效率较低 | 下次可尝试：先生成完整英文文档（不分段），由您本地运行脚本拆分后逐段输入给我进行分段确认 |
| 校对报告与修正文档分两次输出，略有冗余 | 可合并为：一版“带修订标记的文档”（如使用 `~~删除~~` 和 `**新增**` 标记），一版“清洁版” |
| README 和 CHANGELOG 的校对放在最后，略显仓促 | 下次可将它们纳入主文档校对流程，一并处理 |

#### 技术执行层面

| 不足 | 改进建议 |
|:---|:---|
| 首次尝试输出附录时出现了标题重复（`##` 和 `###` 同时存在） | 输出前应检查标题层级连续性，避免跳级或重复 |
| 英文翻译中遗漏了“热力图颜色条”的一段中文 | 翻译完成后应做一次“反向回读”——将英文再译回中文，检查是否有遗漏 |
| 部分代码示例的 import 语句处理不一致（早期有，后期删除） | 应在工作开始前统一约定：示例代码是否包含 import |

#### 协作沟通层面

| 不足 | 改进建议 |
|:---|:---|
| 开始时我输出了较长的“确认规则”段落，增加了您的阅读负担 | 下次可简化为：“已理解。请发送文档，收到后我会分步处理。” |
| 中间有一次我试图一次性输出完整章节（第 6 章），导致卡顿 | 应始终遵守“逐章输出”规则，不因内容连贯性好就尝试合并 |

---

### 三、下次工作可采用的模板流程

```
阶段 0：准备
├── 双方确认：输出粒度（逐章/分段）、格式规范（import 是否保留）、校对清单
│
阶段 1：校正
├── 逐章接收原文 → 输出校正版（带修订标记）→ 您确认 → 下一章
│
阶段 2：翻译
├── 逐章接收校正版 → 输出英文版 → 您确认 → 下一章
│
阶段 3：汇总
├── 生成完整中文文档 + 完整英文文档（由您自行合并，或分段输入后由我合并）
│
阶段 4：附属文件
├── README、CHANGELOG、Prompt 等同步处理
│
阶段 5：收尾
├── 工作评价 + Token 统计
```

---

### 四、总结

本次工作是一次**高效的远程协作**。成功的关键在于：

1. **您的前期准备充分**（代码 Prompt、设计文档 Prompt、校对清单）
2. **分阶段推进 + 及时调整策略**
3. **结构化方法**（清单驱动、逐章输出）

最大的教训是：**在 AI 辅助写作/校对场景中，输出长度限制是硬约束，必须从一开始就设计为“小步快跑”模式，而非“一次性完成”模式。**
